In [ ]:
# Imports
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Set the GPU to use, if available


import albumentations as A
import cv2
import datetime
import gc
import glob
import h5py
import imageio
import io
import ipywidgets as widgets
import math
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import neptune.new as neptune
import neptune.types
import numpy as np
import pandas as pd
import pydicom
import random
import re
import scipy.ndimage as ndimage
import seaborn as sns
import shutil
import tempfile
import tensorflow as tf
import tensorflow_mri as tfmri
import tensorflow_mri
import time
from albumentations.augmentations.transforms import ToFloat
from albumentations.core.composition import OneOf
from collections import defaultdict, Counter
from io import BytesIO
from IPython.display import HTML
from IPython.display import clear_output
from IPython.display import display
from ipywidgets import Button
from ipywidgets import fixed
from ipywidgets import interact
from ipywidgets import IntSlider
from ipywidgets import Layout
from itertools import islice
from keras import backend as K
from keras.metrics import MeanIoU
from keras.utils import to_categorical
from layer_util import ResizeAndConcatenate  
from losses import *
from matplotlib import animation
from matplotlib import gridspec
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.widgets import Button as MatplotlibButton
from matplotlib.widgets import Slider
from mpl_toolkits.axes_grid1 import ImageGrid
from neptune.new.integrations.tensorflow_keras import NeptuneCallback
from neptune.types import File
from os.path import exists
from pandas.plotting import table
from pathlib import Path
from PIL import Image
from skimage.measure import label
from scipy.ndimage import binary_closing
from scipy.spatial import distance
from scipy.stats import shapiro, ttest_rel, wilcoxon
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight
from tensorflow.keras import backend as K
from tensorflow.keras import Input
from tensorflow.keras import Model
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import concatenate
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import MaxPooling2D
from tensorflow_mri import metrics as tfmri_metrics
from tensorflow_mri.metrics import F1Score
from tensorflow_mri import resize_with_crop_or_pad
from tqdm import tqdm
from typing import List, Tuple, Dict
from tina_utils import *
# from unet import *
from unet3plus import *
from unet_seg_utils import *
from array_ops import *
from layer_util import *
from layer_util import ResizeAndConcatenate  
from losses import *

In [ ]:
continue_training = False
model_name = 'DSUP1-15'
neptune_token = "eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiI3ZjZmYTAxZS1hYTI5LTRkNjktOWE0My1iM2NlODZkMDc0NWMifQ=="

if continue_training:
    run = neptune.init_run(
        project="Preclinical/Segmentation",
        api_token=neptune_token,
        with_id=model_name,
    )
else:
    run = neptune.init_run(
        project="Preclinical/Segmentation",
        api_token=neptune_token,
    )
    model_name = list(run.__dict__.values())[-10]

In [ ]:
#  Paths to the directories

# dicom_path = '/workspaces/PhD/unet3+/data/raw/TAC_HGK/DICOMS'
# mask_path = '/workspaces/PhD/unet3+/data/raw/TAC_HGK/MASKS'

dicom_path = '/workspaces/PhD/unet3+/data/raw/zhi_TAC_HGK/DICOMS'
mask_path = '/workspaces/PhD/unet3+/data/raw/zhi_TAC_HGK/MASKS'


# List all DICOM files from the DICOM directory and mask directory
file_names_DICOM = [f for f in os.listdir(dicom_path) if f.endswith('.dcm')]
file_names_MASK = [f for f in os.listdir(mask_path) if f.endswith('.dcm')]

# Creating dictionary mappings of base filenames to their full filename
dicom_files = {os.path.splitext(f)[0]: f for f in file_names_DICOM}
mask_files = {os.path.splitext(f)[0]: f for f in file_names_MASK}


# Print the number of mappings created
print(f"dicom_files: {len(dicom_files)} entries")
print(f"mask_files: {len(mask_files)} entries")

# Matching DICOM and mask files by their base filenames
matched_files = [(dicom_files[k], mask_files[k]) for k in dicom_files if k in mask_files]

# Checking if all files have a matching pair
if len(matched_files) != len(dicom_files) or len(matched_files) != len(mask_files):
    print("Warning: Not all files have a matching pair.")

# Check if there are any mismatches between DICOM and MASK files
unmatched_dicoms = [k for k in dicom_files if k not in mask_files]
unmatched_masks = [k for k in mask_files if k not in dicom_files]

# Print results of the matching check
if not unmatched_dicoms and not unmatched_masks:
    print("All DICOM files have corresponding MASK files.")
else:
    if unmatched_dicoms:
        print(f"Warning: {len(unmatched_dicoms)} DICOM files do not have a corresponding MASK file.")
        print("Unmatched DICOM files (sample):", unmatched_dicoms[:5])
    if unmatched_masks:
        print(f"Warning: {len(unmatched_masks)} MASK files do not have a corresponding DICOM file.")
        print("Unmatched MASK files (sample):", unmatched_masks[:5])


In [ ]:
def extract_details_for_sorting(filename):
    base = filename.split('.')[0]
    parts = re.split(r'[_-]', base)
    
    # Find index of slice part (starts with 'sl')
    slice_idx = next(i for i, p in enumerate(parts) if p.startswith('sl'))
    frame_idx = next(i for i, p in enumerate(parts) if p.startswith('fr'))
    
    # Patient ID is everything before the slice part
    patient_id = '_'.join(parts[:slice_idx])
    
    slice_number = int(parts[slice_idx].replace('sl', ''))
    frame_number = int(parts[frame_idx].replace('fr', ''))
    
    return patient_id, slice_number, frame_number

# Sort the matched files
matched_files_sorted = sorted(
    matched_files,
    key=lambda x: extract_details_for_sorting(x[0])
)

In [ ]:
# Collect unique patient IDs in each folder
dicom_patient_ids = set(
    extract_details_for_sorting(f)[0] for f in file_names_DICOM
)
mask_patient_ids = set(
    extract_details_for_sorting(f)[0] for f in file_names_MASK
)

# Compute intersections and differences
patients_in_both = dicom_patient_ids & mask_patient_ids
patients_only_in_dicom = dicom_patient_ids - mask_patient_ids
patients_only_in_mask = mask_patient_ids - dicom_patient_ids
all_unique_patients = dicom_patient_ids | mask_patient_ids

# Print summary
print(f"\nTotal unique patient IDs in DICOM folder: {len(dicom_patient_ids)}")
print(f"Total unique patient IDs in MASK folder: {len(mask_patient_ids)}")
print(f"Total unique patient IDs across both folders: {len(all_unique_patients)}")

print(f"\nPatient IDs present in BOTH folders ({len(patients_in_both)}):")
for pid in sorted(patients_in_both):
    print(pid)

if patients_only_in_dicom:
    print(f"\nPatient IDs ONLY in DICOM folder ({len(patients_only_in_dicom)}):")
    for pid in sorted(patients_only_in_dicom):
        print(pid)

if patients_only_in_mask:
    print(f"\nPatient IDs ONLY in MASK folder ({len(patients_only_in_mask)}):")
    for pid in sorted(patients_only_in_mask):
        print(pid)


# Define helper function to check duplicate (slice, frame) per patient
def check_duplicate_slices_frames(file_list, label):
    patient_combos = defaultdict(list)

    # Collect all (slice, frame) per patient
    for fname in file_list:
        patient_id, slice_number, frame_number = extract_details_for_sorting(fname)
        patient_combos[patient_id].append((slice_number, frame_number))

    # Check for duplicates
    duplicates_found = False
    print(f"\nChecking for duplicate combinations in {label} files...")
    for patient_id, combos in sorted(patient_combos.items()):
        counter = Counter(combos)
        dupes = [(s, f, count) for (s, f), count in counter.items() if count > 1]
        if dupes:
            duplicates_found = True
            print(f"\nPatient {patient_id} has duplicate entries for:")
            for s, f, count in dupes:
                print(f"  slice {s}, frame {f}: {count} times")

    if not duplicates_found:
        print(f"No duplicate combinations found in {label} files.")


# Run the duplicate checks
check_duplicate_slices_frames(file_names_DICOM, "DICOM")
check_duplicate_slices_frames(file_names_MASK, "MASK")


In [ ]:
# Resizing images, if needed
SIZE_X = 256
SIZE_Y = 256

# Number of classes for segmentation
n_classes = 3 

In [ ]:
# Checking whether a mask has ROI 
def check_for_roi(mask_path, mask_name):
    dsM = pydicom.dcmread(os.path.join(mask_path, mask_name))
    mask = dsM.pixel_array
    # Assuming non-zero values in the mask indicate ROI
    return np.any(mask > 0)

In [ ]:
def load_process_data_with_roi(file_pairs, dicom_path, mask_path, SIZE_X, SIZE_Y):
    images, masks, processed_filenames = [], [], []
    all_frames_test_images = {}  # Store all frames (all slices, all time points) for all patients
    relevant_frames = {}  # Store only EDV & ESV frames for cardiac function calculations
    patient_data = {}
    resizing_logs = []

    # Group the files by patient ID
    for dicom_name, mask_name in file_pairs:
        patient_id, slice_number, frame_number = extract_details_for_sorting(dicom_name)

        if patient_id not in patient_data:
            patient_data[patient_id] = {'frames': {}}

        patient_data[patient_id]['frames'].setdefault(frame_number, []).append((dicom_name, mask_name))

    total_patients = 0
    total_slices = 0
    total_frames_before = 0
    total_frames_after = 0

    # Process each patient
    for patient_id, data in patient_data.items():
        frames_with_roi = []
        patient_frames = {}  # Store all frames (slice-wise) for this patient
        in_flow_artifact_frames = []  # Store frames with in-flow artifacts

        # Identify frames that contain segmentation (ROI)
        for frame_number, slices in data['frames'].items():
            has_roi = any([check_for_roi(mask_path, mask_name) for _, mask_name in slices])

            if has_roi:
                frames_with_roi.append(frame_number)

        # Identify **EDV & ESV frames**
        if len(frames_with_roi) >= 2:
            selected_frames = sorted(frames_with_roi[:2])  # First two frames with ROI -> EDV & ESV
            relevant_frames[patient_id] = selected_frames  # Store for cardiac function calculation
        else:
            print(f"Warning: Patient {patient_id} has {len(frames_with_roi)} frames with ROI, expected 2.")
            continue  # Skip patients with insufficient ROI frames

        # Detect additional frames containing **in-flow artifacts**
        for frame_number in frames_with_roi:
            if frame_number not in selected_frames:
                in_flow_artifact_frames.append(frame_number)

        # Train using **all frames with segmentation** (EDV, ESV, + in-flow artifacts)
        all_training_frames = selected_frames + in_flow_artifact_frames

        original_num_frames = len(data['frames'])
        original_num_slices = len(data['frames'][selected_frames[0]])
        total_slices += original_num_slices
        total_frames_before += original_num_frames

        # Store all frames for every patient in the correct format
        for frame_number in sorted(data['frames'].keys()):  # Iterate over all frames (not just EDV/ESV)
            frame_images = []  # Store all slices for this frame

            for dicom_name, _ in data['frames'][frame_number]:  # only need DICOMs, so ignore masks
                try:
                    # Read DICOM image
                    dsI = pydicom.dcmread(os.path.join(dicom_path, dicom_name))
                    curImage = dsI.pixel_array

                    # Resize if needed
                    if curImage.shape[0] != SIZE_X or curImage.shape[1] != SIZE_Y:
                        curImage = tf.image.resize_with_crop_or_pad(
                            curImage[:, :, np.newaxis], SIZE_X, SIZE_Y
                        ).numpy()[:, :, 0]

                    frame_images.append(curImage)

                except Exception as e:
                    print(f"Error processing {dicom_name}: {e}")

            # Store slices per frame
            if frame_images:
                patient_frames[frame_number] = np.stack(frame_images, axis=0)  # Shape: (S, H, W)




        # Convert dictionary to sorted list of frames (x, y, z, t)
        if patient_frames:
            sorted_frame_numbers = sorted(patient_frames.keys())  # Ensure correct frame order
            # Original shape: (F, S, H, W)
            data_f_s_h_w = np.stack([patient_frames[f] for f in sorted_frame_numbers], axis=0)
                    
            # New shape: (H, W, S, F) → (x, y, z, t)
            data_h_w_s_f = np.transpose(data_f_s_h_w, (2, 3, 1, 0))
            all_frames_test_images[patient_id] = data_h_w_s_f



        # Process all slices for the selected training frames
        for frame_number in all_training_frames:
            for dicom_name, mask_name in data['frames'][frame_number]:
                try:
                    # Read DICOM and mask
                    dsI = pydicom.dcmread(os.path.join(dicom_path, dicom_name))
                    dsM = pydicom.dcmread(os.path.join(mask_path, mask_name))

                    curImage = dsI.pixel_array
                    curMask = dsM.pixel_array

                    if curImage.shape != curMask.shape:
                        print(f"Shape mismatch for {dicom_name}: {curImage.shape} vs {curMask.shape}")
                        continue

                    # Resize if needed
                    if curImage.shape[0] != SIZE_X or curImage.shape[1] != SIZE_Y:
                        curImage = tf.image.resize_with_crop_or_pad(curImage[:, :, np.newaxis], SIZE_X, SIZE_Y).numpy()[:, :, 0]
                        curMask = tf.image.resize_with_crop_or_pad(curMask[:, :, np.newaxis], SIZE_X, SIZE_Y).numpy()[:, :, 0]

                    # Append processed images and masks
                    images.append(curImage)
                    masks.append(curMask)
                    processed_filenames.append(dicom_name)

                except Exception as e:
                    print(f"Error processing {dicom_name}: {e}")

        filtered_num_frames = len(all_training_frames)
        total_frames_after += filtered_num_frames

        # Print debugging information
        print(f"\nPatient {patient_id}: Selected Frames with ROI: {selected_frames} (EDV & ESV)")
        print(f"Additional frames with in-flow artifacts: {in_flow_artifact_frames}")
        print(f"Total slices: {original_num_slices}")
        print(f"Shape (H, W, S, F): {data_h_w_s_f.shape}")
        print(f"Total Frames Before: {original_num_frames}, Total Frames After: {filtered_num_frames}")

    images = np.array(images)
    masks = np.array(masks)

    return images, masks, processed_filenames, relevant_frames, all_frames_test_images, patient_data



# Load the dataset while storing all patient frames
images, masks, processed_filenames, relevant_frames, all_frames_test_images, patient_data = load_process_data_with_roi(
    matched_files_sorted, dicom_path, mask_path, SIZE_X, SIZE_Y
)


# Print verification
print(f"Total patients stored in all_frames_test_images: {len(all_frames_test_images)}")
for patient_id, frames in all_frames_test_images.items():
    print(f"Patient {patient_id}: {len(frames)} frames saved")


# Verify that relevant_frames is a dictionary
print("Type of relevant_frames after preprocessing:", type(relevant_frames))
print("Content of relevant_frames after preprocessing:", relevant_frames)

In [ ]:
# Set a higher embedding limit for animations (in MB)
plt.rcParams['animation.embed_limit'] = 300  
# Adjusting figure size and DPI for better resolution
fig, ax = plt.subplots(figsize=(10, 10), dpi=100)
plt.axis('off')
plt.close()

# Combine all DICOM and mask data into a single array with associated metadata
all_dicom_images = []
metadata = []

# Dictionary to store images and metadata by patient and slice
patient_slices = {}

# Collecting data from the processed DICOM and mask arrays
for idx in range(len(processed_filenames)):
    dicom_image = images[idx]
    mask_image = masks[idx]
    
    # Extract patient ID, slice, and frame info from the DICOM filename
    dicom_name = processed_filenames[idx]
    patient_id, slice_number, frame_number = extract_details_for_sorting(dicom_name)
    
    # Group the data by patient and slice
    if patient_id not in patient_slices:
        patient_slices[patient_id] = {}
    if slice_number not in patient_slices[patient_id]:
        patient_slices[patient_id][slice_number] = []
    
    # Store the combined DICOM and mask image along with frame metadata
    combined_image = np.concatenate((dicom_image, mask_image), axis=0)
    patient_slices[patient_id][slice_number].append((combined_image, frame_number))

# Organize all data for animation: diastolic then systolic for each slice, then move to the next patient
for patient_id in sorted(patient_slices.keys()):
    for slice_number in sorted(patient_slices[patient_id].keys()):
        # Sort frames within each slice (adiastolic frame first then systolic)-recheck this if frame start with systolic!
        frames_sorted = sorted(patient_slices[patient_id][slice_number], key=lambda x: x[1])
        for combined_image, frame_number in frames_sorted:
            all_dicom_images.append(combined_image)
            metadata.append((patient_id, slice_number, frame_number))

# Convert lists to numpy arrays for easy indexing
all_dicom_images = np.array(all_dicom_images)

# Initialize display using the first combined image
im = ax.imshow(all_dicom_images[0], cmap='gray')
title = ax.set_title(f"Patient ID: {metadata[0][0]}, Slice: {metadata[0][1]}, Frame: {metadata[0][2]}")

# Function to initialize the animation
def init():
    im.set_data(all_dicom_images[0])
    title.set_text(f"Patient ID: {metadata[0][0]}, Slice: {metadata[0][1]}, Frame: {metadata[0][2]}")
    return [im, title]

# Function to animate the frames
def animate(i):
    im.set_data(all_dicom_images[i])
    title.set_text(f"Patient ID: {metadata[i][0]}, Slice: {metadata[i][1]}, Frame: {metadata[i][2]}")
    return [im, title]

# Create the animation
anim = animation.FuncAnimation(fig, animate, init_func=init, frames=len(all_dicom_images), interval=700, blit=True)

# Convert the animation to an HTML5 video and embed it
html_video = anim.to_html5_video()
html_video = html_video.replace('<video', '<video style="max-width: 100%; height: auto;"')

# Display the animation
HTML(html_video)


In [ ]:
# Simplify masks function
def simplify_masks(masks):
    # Create an empty array to hold the simplified mask
    masks = np.round(masks).astype(np.uint16)  # Ensure integer values
    simplified_masks = np.zeros_like(masks, dtype=np.uint8)
    
    # Define conditions for each class
    simplified_masks[masks == 0] = 0  # Background
    simplified_masks[(masks > 10) & (masks < 200)] = 1  # Myocardium
    simplified_masks[masks > 200] = 2  # Blood pool

    unique_values_after = np.unique(simplified_masks)
    print(f"Unique values after simplifying: {unique_values_after}")
    
    return simplified_masks

In [ ]:
# Debug-checking Pixel Values in the Mask 

# Select a sample patient ID, frame, and slice for analysis
sample_patient_id = list(patient_slices.keys())[1]  # Choose a patient    
sample_slice = sorted(patient_slices[sample_patient_id].keys())[5]  # Choose a slice
sample_frames = sorted(patient_slices[sample_patient_id][sample_slice], key=lambda x: x[1])  # Sort the frames

# Extract the dicom and mask images for the selected patient
sample_combined_image, sample_frame_number = sample_frames[1]  # First frame for that slice
sample_dicom_image = sample_combined_image[:SIZE_Y, :]  # Extracting the DICOM part
sample_mask_image = sample_combined_image[SIZE_Y:, :]  # Extracting the mask part

# Simplify the mask
simplified_mask = simplify_masks(sample_mask_image)

# Flatten the mask for histogram analysis
flattened_mask = sample_mask_image.flatten()

# Plot histogram of pixel values in the mask
plt.figure(figsize=(10, 6))
plt.hist(flattened_mask, bins=100, color='blue', alpha=0.7)
plt.title(f"Histogram of Pixel Values in the Mask (Patient ID: {sample_patient_id})")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

# Print unique values in the simplified mask
unique_values = np.unique(simplified_mask)
print(f"Unique values in the simplified mask for Patient {sample_patient_id}: {unique_values}")


# Plot the original mask and the simplified classes
plt.figure(figsize=(15, 5))

plt.subplot(1, 4, 1)
plt.title("Original Mask")
plt.imshow(sample_mask_image, cmap='viridis')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title("Background (Class 0)")
plt.imshow(simplified_mask == 0, cmap='viridis')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title("Myocardium (Class 1)")
plt.imshow(simplified_mask == 1, cmap='viridis')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title("Blood Pool (Class 2)")
plt.imshow(simplified_mask == 2, cmap='viridis')
plt.axis('off')

plt.show()


In [ ]:
# Extract the regions for each class based on the simplified mask
background_pixels = sample_mask_image[simplified_mask == 0]
myocardium_pixels = sample_mask_image[simplified_mask == 1]
blood_pool_pixels = sample_mask_image[simplified_mask == 2]

# Plot histograms for each class
plt.figure(figsize=(18, 6))

# Histogram for Background (Class 0)
plt.subplot(1, 3, 1)
plt.hist(background_pixels.flatten(), bins=100, color='blue', alpha=0.7)
plt.title("Histogram of Pixel Values for Background (Class 0)")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.grid(True)

# Histogram for Myocardium (Class 1)
plt.subplot(1, 3, 2)
plt.hist(myocardium_pixels.flatten(), bins=100, color='green', alpha=0.7)
plt.title("Histogram of Pixel Values for Myocardium (Class 1)")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.grid(True)

# Histogram for Blood Pool (Class 2)
plt.subplot(1, 3, 3)
plt.hist(blood_pool_pixels.flatten(), bins=100, color='red', alpha=0.7)
plt.title("Histogram of Pixel Values for Blood Pool (Class 2)")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Debug-checking for abnormal pixel values

# Set threshold ranges for abnormal pixel values
lower_abnormal_range = (1, 100)  # Between 1 and 100 is unexpected-abnormal pixel values
upper_abnormal_range = (400, 500)  # Between 400 and 500 is unexpected-abnormal pixel values

# List to store patient IDs with abnormal pixel values
abnormal_patients = []

# Loop over each patient to flag abnormalities first (no plotting yet)
for patient_id in sorted(patient_slices.keys()):
    all_flattened_masks = []
    
    for slice_number in sorted(patient_slices[patient_id].keys()):
        frames_sorted = sorted(patient_slices[patient_id][slice_number], key=lambda x: x[1])
        
        for combined_image, frame_number in frames_sorted:
            mask_image = combined_image[SIZE_Y:, :]  # Extracting the mask part
            flattened_mask = mask_image.flatten()
            all_flattened_masks.append(flattened_mask)
    
    # Combine all masks for this patient
    combined_flattened_mask = np.concatenate(all_flattened_masks)
    
    # Check if there are abnormal pixel values in the mask
    has_lower_abnormal_pixels = np.any((combined_flattened_mask > lower_abnormal_range[0]) & (combined_flattened_mask < lower_abnormal_range[1]))
    has_upper_abnormal_pixels = np.any((combined_flattened_mask > upper_abnormal_range[0]) & (combined_flattened_mask < upper_abnormal_range[1]))

    # If abnormal pixel values are detected, add patient ID to the abnormal list
    if has_lower_abnormal_pixels or has_upper_abnormal_pixels:
        abnormal_patients.append(patient_id)

# Print the list of patients with abnormal pixel values (before any plotting)
if abnormal_patients:
    print("\nList of Patients with Abnormal Pixel Values:")
    for patient in abnormal_patients:
        print(f"Patient ID: {patient}")
else:
    print("\nNo abnormal pixel values found for any patient.")


# Generate histograms and plots for abnormal patients only
for patient_id in abnormal_patients:
    all_flattened_masks = []
    
    for slice_number in sorted(patient_slices[patient_id].keys()):
        frames_sorted = sorted(patient_slices[patient_id][slice_number], key=lambda x: x[1])
        
        for combined_image, frame_number in frames_sorted:
            mask_image = combined_image[SIZE_Y:, :]  # Extracting the mask part
            flattened_mask = mask_image.flatten()
            all_flattened_masks.append(flattened_mask)
    
    # Combine all masks for this patient with abnormal pixel values
    combined_flattened_mask = np.concatenate(all_flattened_masks)
    
    # Generate histogram for the abnormal patient
    unique_values = np.unique(combined_flattened_mask)
    print(f"\nAbnormal pixel values detected for Patient {patient_id}. Unique pixel values: {unique_values}")
    
    plt.figure(figsize=(10, 6))
    plt.hist(combined_flattened_mask, bins=100, color='blue', alpha=0.7)
    plt.title(f"Abnormal Histogram of Pixel Values for Patient {patient_id}")
    plt.xlabel("Pixel Value")
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.show()

    # Plot original mask and simplified classes for the first slice and frame
    for slice_number in sorted(patient_slices[patient_id].keys()):
        combined_image, frame_number = patient_slices[patient_id][slice_number][1]  # First frame for that slice
        mask_image = combined_image[SIZE_Y:, :]  # Extract the mask part
        simplified_mask = simplify_masks(mask_image)

        # Plot the original mask and the simplified classes
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 4, 1)
        plt.title("Original Mask")
        plt.imshow(mask_image, cmap='viridis')
        plt.axis('off')

        plt.subplot(1, 4, 2)
        plt.title("Background (Class 0)")
        plt.imshow(simplified_mask == 0, cmap='viridis')
        plt.axis('off')

        plt.subplot(1, 4, 3)
        plt.title("Myocardium (Class 1)")
        plt.imshow(simplified_mask == 1, cmap='viridis')
        plt.axis('off')

        plt.subplot(1, 4, 4)
        plt.title("Blood Pool (Class 2)")
        plt.imshow(simplified_mask == 2, cmap='viridis')
        plt.axis('off')

        plt.show()

        # Stop after visualizing the first slice and frame to avoid over-plotting
        break


In [ ]:
# List of patients to exclude (to be removed from all datasets) 78-4=74       78-5=73
# excluded_patient = S_2012121107 (HGK)   s_2012071501_c5r2 (TAC) 
# Abnormal pixel Value: S_2012110505 S_2012121004


excluded_patient = ['S_2012121107', 's_2012071501_c5r2', 'S_2012110505', 'S_2012121004', 'S_2012121106']

# Extract and sort unique patients from filenames
all_patients = sorted(set(extract_details_for_sorting(filename)[0] for filename in processed_filenames))

# Exclude specified patients
patients = [patient for patient in all_patients if patient not in excluded_patient]

# Shuffle the patients to randomize the splits
np.random.seed(42)  # Fix seed for reproducibility
np.random.shuffle(patients)

# Specify fixed numbers for the train/val/test splits
num_train = 50
num_val = 8
num_test = 15

# Ensure total matches number of patients
assert num_train + num_val + num_test == len(patients), "Mismatch in total patient numbers!"

# Assign patients to train, validation, and test sets
train_patients = patients[:num_train]
val_patients = patients[num_train:num_train + num_val]
test_patients = patients[num_train + num_val:num_train + num_val + num_test]

# Now filter the images, masks, and filenames into the appropriate sets based on patient ID
train_images, train_masks, val_images, val_masks, test_images, test_masks = [], [], [], [], [], []
train_metadata, val_metadata, test_metadata = [], [], []

for image, mask, filename in zip(images, masks, processed_filenames):
    patient_id, slice_id, frame_id = extract_details_for_sorting(filename)

    # Skip excluded patients
    if patient_id in excluded_patient:
        continue

    # Assign images and masks to the corresponding dataset based on patient ID
    if patient_id in train_patients:
        train_images.append(image)
        train_masks.append(mask)
        train_metadata.append((patient_id, slice_id, frame_id))

    elif patient_id in val_patients:
        val_images.append(image)
        val_masks.append(mask)
        val_metadata.append((patient_id, slice_id, frame_id))

    elif patient_id in test_patients:
        test_images.append(image)
        test_masks.append(mask)
        test_metadata.append((patient_id, slice_id, frame_id))

# Convert lists to numpy arrays for shape printing
train_images = np.array(train_images)
val_images = np.array(val_images)
test_images = np.array(test_images)

train_masks = np.array(train_masks)
val_masks = np.array(val_masks)
test_masks = np.array(test_masks)

# Print dataset distribution
print("Dataset distribution:")
print("Number of patients in training set:", len(train_patients))
print("Number of patients in validation set:", len(val_patients))
print("Number of patients in test set:", len(test_patients))

# Print the shapes and data types of the image and mask sets
print("\nShapes and data types of data:")
print("Training set images shape:", train_images.shape)
print("Training set masks shape:", train_masks.shape)
print("Validation set images shape:", val_images.shape)
print("Validation set masks shape:", val_masks.shape)
print("Test set images shape:", test_images.shape)
print("Test set masks shape:", test_masks.shape)

# Print unique patient IDs in each set vertically
print("\nUnique patient IDs in training set:")
for patient in sorted(train_patients):
    print(patient)

print("\nUnique patient IDs in validation set:")
for patient in sorted(val_patients):
    print(patient)

print("\nUnique patient IDs in test set:")
for patient in sorted(test_patients):
    print(patient)

In [ ]:
def normalize_images(images):
    # cast to float64 for a higher level of precision during calculations
    images = tf.cast(images, tf.float64)
    min_val = tf.reduce_min(images, axis=[1, 2], keepdims=True)
    max_val = tf.reduce_max(images, axis=[1, 2], keepdims=True)

    # Check if the image has variable pixel values for normalization
    normalized_images = tf.where(
        max_val > min_val,
        (images - min_val) / (max_val - min_val),
        images
    )

    return normalized_images

In [ ]:
def check_normalization(images, dataset_name):
    all_good = True
    for i, img in enumerate(images):
        img_min = tf.reduce_min(img).numpy()
        img_max = tf.reduce_max(img).numpy()
        
        if img_min != 0 or img_max != 1:
            print(f"Warning: Image {i} in {dataset_name} is not normalized correctly (min: {img_min}, max: {img_max})")
            all_good = False
    
    if all_good:
        print(f"All images in {dataset_name} are correctly normalized (min: 0, max: 1).")

In [ ]:
# Albumentations pipeline 

augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.RandomRotate90(p=0.5),  # Rotate 90 degrees randomly
    A.RandomBrightnessContrast(brightness_limit=0.17, contrast_limit=0.0, p=0.4),
    A.RandomGamma(gamma_limit=(70, 120), p=0.3),  # Gamma correction
    A.GaussNoise(var_limit=(30.0, 100.0), p=0.5),  # Stronger acquisition noise simulation
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=10, p=0.5),  # Geometric transformations 
])

In [ ]:
def data_generator_with_augmentation(images, masks, batch_size):
    while True:
        indices = np.random.choice(len(images), batch_size)
        batch_images = []
        batch_masks = []
        for i in indices:
            # Convert images to uint8 (0-255) for augmentation
            img = (images[i] * 255).astype(np.uint8)
            mask = masks[i].astype(np.int32)  # Masks as integers
            augmented = augmentation_pipeline(image=img, mask=mask)
            
            # Add the augmented image and mask to the batch
            batch_images.append(augmented['image']) 
            batch_masks.append(augmented['mask'])
        
        # Apply the normalize_images function to the entire batch
        batch_images = normalize_images(np.array(batch_images, dtype=np.float32))
        
        yield batch_images.numpy(), np.array(batch_masks, dtype=np.int32)

In [ ]:
# Apply normalization to the datasets
train_images = normalize_images(train_images)
val_images = normalize_images(val_images)
test_images = normalize_images(test_images)

# Verify all images in each dataset are correctly normalized (min: 0, max: 1) 
check_normalization(train_images, "Train images")
check_normalization(val_images, "Val images")
check_normalization(test_images, "Test images")

In [ ]:
# Simplify masks after image normalization
train_masks = simplify_masks(train_masks)
val_masks = simplify_masks(val_masks)
test_masks = simplify_masks(test_masks)

print("Unique values in train_masks:", np.unique(train_masks))
print("Unique values in val_masks:", np.unique(val_masks))
print("Unique values in test_masks:", np.unique(test_masks))

In [ ]:
def check_segmentation_consistency(masks, metadata, dataset_name=""):
    """
    Check masks for missing myocardium or blood pool between slices.

    Args:
    - masks: Simplified mask array of shape (N, H, W) for all images.
    - metadata: List of tuples (patient_id, slice_id, frame_id).
    - dataset_name: Name of the dataset (e.g., "train", "val", "test") for printing.
    """
    from collections import defaultdict

    # Organize masks and metadata by patient and frame
    patient_frames = defaultdict(lambda: defaultdict(list))

    for i, (patient_id, slice_id, frame_id) in enumerate(metadata):
        patient_frames[patient_id][frame_id].append((slice_id, masks[i]))

    # Variable to track problematic patients
    issues_found = False
    problematic_patients = {}

    # Iterate over each patient and frame to check consistency
    for patient_id, frames in patient_frames.items():
        for frame_id, slices in sorted(frames.items()):
            slices.sort(key=lambda x: x[0])  # Sort by slice ID

            for i in range(1, len(slices) - 1):  # Skip first and last slices
                prev_mask = slices[i - 1][1]
                cur_mask = slices[i][1]
                next_mask = slices[i + 1][1]

                # Check for myocardium (label 1) and blood pool (label 2)
                missing_myocardium = (
                    np.any(prev_mask == 1) and np.any(next_mask == 1) and not np.any(cur_mask == 1)
                )
                missing_blood_pool = (
                    np.any(prev_mask == 2) and np.any(next_mask == 2) and not np.any(cur_mask == 2)
                )

                if missing_myocardium or missing_blood_pool:
                    issues_found = True
                    if patient_id not in problematic_patients:
                        problematic_patients[patient_id] = []
                    problematic_patients[patient_id].append(
                        f"  Frame: {frame_id} - Slice {slices[i][0]} "
                        f"(Missing Myocardium: {missing_myocardium}, Missing Blood Pool: {missing_blood_pool})"
                    )

    # Print summary for the dataset
    print(f"\nChecking {dataset_name} masks for missing myocardium/blood pool...")
    if issues_found:
        print("Issues found in the following patients!:")
        for patient_id, issues in problematic_patients.items():
            print(f"Patient: {patient_id}")
            for issue in issues:
                print(issue)
    else:
        print(f"No missing myocardium/blood pool in {dataset_name} masks")


check_segmentation_consistency(train_masks, train_metadata, dataset_name="train")
check_segmentation_consistency(val_masks, val_metadata, dataset_name="validation")
check_segmentation_consistency(test_masks, test_metadata, dataset_name="test")

In [ ]:
def check_segmentation_consistency(masks, metadata, dataset_name=""):
    """
    Check masks for missing myocardium or blood pool between slices,
    and check most basal slice consistency across ED and ES.

    Args:
    - masks: Simplified mask array of shape (N, H, W) for all images.
    - metadata: List of tuples (patient_id, slice_id, frame_id).
    - dataset_name: Name of the dataset (e.g., "train", "val", "test") for printing.
    """
    import numpy as np
    from collections import defaultdict

    # Organize masks and metadata by patient and frame
    patient_frames = defaultdict(lambda: defaultdict(list))
    slice_lookup = defaultdict(lambda: defaultdict(dict))  # patient -> frame -> slice -> mask

    for i, (patient_id, slice_id, frame_id) in enumerate(metadata):
        patient_frames[patient_id][frame_id].append((slice_id, masks[i]))
        slice_lookup[patient_id][frame_id][slice_id] = masks[i]

    issues_found = False
    problematic_patients = {}

    for patient_id, frames in patient_frames.items():
        for frame_id, slices in sorted(frames.items()):
            slices.sort(key=lambda x: x[0])  # Sort by slice ID

            for i in range(1, len(slices) - 1):  # Skip first and last
                prev_mask = slices[i - 1][1]
                cur_mask = slices[i][1]
                next_mask = slices[i + 1][1]

                missing_myocardium = (
                    np.any(prev_mask == 1) and np.any(next_mask == 1) and not np.any(cur_mask == 1)
                )
                missing_blood_pool = (
                    np.any(prev_mask == 2) and np.any(next_mask == 2) and not np.any(cur_mask == 2)
                )

                if missing_myocardium or missing_blood_pool:
                    issues_found = True
                    if patient_id not in problematic_patients:
                        problematic_patients[patient_id] = []
                    problematic_patients[patient_id].append(
                        f"  Frame: {frame_id} - Slice {slices[i][0]} "
                        f"(Missing Myocardium: {missing_myocardium}, Missing Blood Pool: {missing_blood_pool})"
                    )

        # === Basal Slice Check ===
        ed_frame = frames.get(0, [])
        es_frame = frames.get(1, [])

        # Only proceed if both frames are present
        if not ed_frame or not es_frame:
            continue

        # Consider only first and last slice for basal candidate
        slice_ids = sorted({sid for sid, _ in ed_frame + es_frame})
        if not slice_ids:
            continue

        first_sid = slice_ids[0]
        last_sid = slice_ids[-1]

        def blood_pool_area(mask):
            return np.sum(mask == 2) if mask is not None else 0

        ed_first_mask = slice_lookup[patient_id][0].get(first_sid)
        es_first_mask = slice_lookup[patient_id][1].get(first_sid)
        ed_last_mask = slice_lookup[patient_id][0].get(last_sid)
        es_last_mask = slice_lookup[patient_id][1].get(last_sid)

        first_total_bp = blood_pool_area(ed_first_mask) + blood_pool_area(es_first_mask)
        last_total_bp = blood_pool_area(ed_last_mask) + blood_pool_area(es_last_mask)

        # Determine basal slice as the one with the most blood pool
        if first_total_bp >= last_total_bp:
            basal_sid = first_sid
            ed_mask = ed_first_mask
            es_mask = es_first_mask
        else:
            basal_sid = last_sid
            ed_mask = ed_last_mask
            es_mask = es_last_mask

        has_ed_bp = ed_mask is not None and np.any(ed_mask == 2)
        has_es_bp = es_mask is not None and np.any(es_mask == 2)

        if has_ed_bp != has_es_bp:
            issues_found = True
            if patient_id not in problematic_patients:
                problematic_patients[patient_id] = []
            problematic_patients[patient_id].append(
                f"  Basal slice {basal_sid} has blood pool only in {'ED' if has_ed_bp else 'ES'}"
            )

    # === Final Report ===
    print(f"\nChecking {dataset_name} masks for missing myocardium/blood pool and basal consistency...")
    if issues_found:
        print("Issues found in the following patients!:")
        for patient_id, issues in problematic_patients.items():
            print(f"Patient: {patient_id}")
            for issue in issues:
                print(issue)
    else:
        print(f"No issues in {dataset_name} masks")




check_segmentation_consistency(train_masks, train_metadata, dataset_name="train")
check_segmentation_consistency(val_masks, val_metadata, dataset_name="validation")
check_segmentation_consistency(test_masks, test_metadata, dataset_name="test")

In [ ]:
# # # Delete the artifact using del
# del run["Training GIFs"]

In [ ]:
GIFs

In [ ]:
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation, PillowWriter
# from tqdm import tqdm

# def create_patient_gif_with_dynamic_alpha_safe(images, masks, metadata, save_dir, max_slices_per_row=3, fps=1, dpi=80):
#     os.makedirs(save_dir, exist_ok=True)

#     images = np.array(images)
#     masks = np.array(masks)

#     assert images.shape == masks.shape, "Image and mask shape mismatch"
#     assert all(isinstance(x, tuple) and len(x) == 3 for x in metadata), "Metadata format must be (pid, slice, frame)"

#     # ✅ Build patient-wise frame dictionary
#     patient_data = {}
#     for idx, (pid, sl, fr) in enumerate(metadata):
#         if pid not in patient_data:
#             patient_data[pid] = {}
#         if fr not in patient_data[pid]:
#             patient_data[pid][fr] = []
#         patient_data[pid][fr].append((sl, images[idx], masks[idx]))

#     for pid, frames in tqdm(patient_data.items(), desc="Creating GIFs per patient"):
#         sorted_frames = sorted(frames.items(), key=lambda x: x[0])  # sort by frame
#         all_slices = sorted(set(sl for slices in frames.values() for sl, _, _ in slices))

#         rows = (len(all_slices) + max_slices_per_row - 1) // max_slices_per_row
#         cols = min(len(all_slices), max_slices_per_row)

#         fig, axs = plt.subplots(rows, cols, figsize=(cols * 6, rows * 6), dpi=dpi)
#         axs = axs.flatten()

#         def init():
#             for ax in axs:
#                 ax.axis('off')
#             return axs
        
#         def update(frame_idx):
#             frame_number, slices = sorted_frames[frame_idx]
#             slice_map = {sl: (img, msk) for sl, img, msk in slices}

#             for ax in axs:
#                 ax.clear()
#                 ax.axis('off')

#             for i, sl in enumerate(all_slices):
#                 if sl not in slice_map:
#                     continue

#                 img, msk = slice_map[sl]
#                 assert isinstance(img, np.ndarray) and isinstance(msk, np.ndarray)

#                 # ✅ Binary masks
#                 myocardium_mask = (msk == 1).astype(float)
#                 blood_pool_mask = (msk == 2).astype(float)

#                 # ✅ Dynamic alpha
#                 myocardium_alpha = myocardium_mask * 0.3
#                 blood_pool_alpha = blood_pool_mask * 0.3

#                 axs[i].imshow(img, cmap='gray')
#                 axs[i].imshow(myocardium_mask, cmap='Blues', alpha=myocardium_alpha)
#                 axs[i].imshow(blood_pool_mask, cmap='jet', alpha=blood_pool_alpha)
#                 axs[i].set_title(f"Slice {sl}, Frame {frame_number}", fontsize=14)

#             fig.suptitle(f"{pid} — Frame {frame_number}", fontsize=20)
#             return axs


#         anim = FuncAnimation(fig, update, init_func=init, frames=len(sorted_frames), interval=1000 // fps, blit=False)

#         gif_path = os.path.join(save_dir, f"{pid}_dynamic_alpha.gif")
#         anim.save(gif_path, writer=PillowWriter(fps=fps), dpi=dpi)
#         plt.close(fig)


#         run[f"Training GIFs/{pid}"].upload(gif_path)
#         print(f"Saved GIF for Patient {pid}: {gif_path}")


# save_dir = "/workspaces/PhD/outputs/SEGtest/GIFs_Dynamic_Alpha_Safe"
# create_patient_gif_with_dynamic_alpha_safe(train_images, train_masks, train_metadata, save_dir)



In [ ]:
new

In [ ]:
def save_preprocessed_patient_data(images, masks, metadata, save_dir):
    """
    Save preprocessed images and masks per patient with correct shape (Height, Width, Slices, Frames),
    ensuring consistent slice order across frames using slice_id sorting.
    """
    os.makedirs(save_dir, exist_ok=True)

    patient_data = {}

    # Group all data by patient
    for img, mask, meta in zip(images, masks, metadata):
        patient_id, slice_id, frame_id = meta

        if patient_id not in patient_data:
            patient_data[patient_id] = {"images": {}, "masks": {}}

        if frame_id not in patient_data[patient_id]["images"]:
            patient_data[patient_id]["images"][frame_id] = []
            patient_data[patient_id]["masks"][frame_id] = []

        # Store (slice_id, img/mask) so we can sort later
        patient_data[patient_id]["images"][frame_id].append((slice_id, img))
        patient_data[patient_id]["masks"][frame_id].append((slice_id, mask))

    for patient_id, data in patient_data.items():
        patient_dir = os.path.join(save_dir, patient_id)
        os.makedirs(patient_dir, exist_ok=True)

        all_frames = []
        all_masks = []

        for frame_id in sorted(data["images"].keys()):
            # Sort slices by slice_id before stacking
            sorted_imgs = sorted(data["images"][frame_id], key=lambda x: x[0])
            sorted_masks = sorted(data["masks"][frame_id], key=lambda x: x[0])

            frame_images = np.stack([img for _, img in sorted_imgs], axis=-1)  # (H, W, S)
            frame_masks = np.stack([mask for _, mask in sorted_masks], axis=-1)

            all_frames.append(frame_images)
            all_masks.append(frame_masks)

        # Final stack across time axis → shape (H, W, S, F)
        images_np = np.stack(all_frames, axis=-1)
        masks_np = np.stack(all_masks, axis=-1)

        # Save
        np.save(os.path.join(patient_dir, "images.npy"), images_np)
        np.save(os.path.join(patient_dir, "masks.npy"), masks_np)

        print(f"✅ Saved preprocessed data for Patient {patient_id} in {patient_dir}")
        print(f"   Images shape: {images_np.shape}")
        print(f"   Masks shape:  {masks_np.shape}")


In [ ]:
# Save preprocessed data per patient
save_dir = "/workspaces/PhD/unet3+/data/clean/SEG420"
os.makedirs(save_dir, exist_ok=True)  

# Save preprocessed data per patient without adding a channel dimension
save_preprocessed_patient_data(test_images, test_masks, test_metadata, os.path.join(save_dir, "test"))

In [ ]:

def plot_mid_slice_from_saved_data(save_dir):
    """
    Plot the mid-slice for all frames of the first patient in the saved directory.
    Uses contrast scaling (vmin=0, vmax=1) assuming normalized images.
    """
    # Get the first patient directory
    patient_dirs = sorted(os.listdir(save_dir))
    if not patient_dirs:
        print("No patient data found in the specified directory.")
        return

    first_patient_id = patient_dirs[0]
    patient_dir = os.path.join(save_dir, first_patient_id)

    # Load the reshaped images
    reshaped_frames_array = np.load(os.path.join(patient_dir, "images.npy"))  # Shape: (H, W, S, F)
    H, W, S, F = reshaped_frames_array.shape

    print(f"Plotting mid-slice for the first patient: {first_patient_id}")
    print(f"  Loaded Images shape: {H, W, S, F}")

    # Get the index of the mid-slice
    mid_slice_index = S // 2

    # Extract the mid-slice across all frames
    mid_slice_frames = reshaped_frames_array[:, :, mid_slice_index, :]  # Shape: (H, W, F)

    # Determine the grid size for subplots
    n_cols = min(5, F)  # Limit the number of columns to 5
    n_rows = (F + n_cols - 1) // n_cols  # Calculate rows needed to fit all frames

    # Create a grid of subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    fig.suptitle(f"Patient {first_patient_id} - Mid-Slice Across Frames", fontsize=16)

    axes = axes.ravel()

    for f in range(F):
        ax = axes[f]
        ax.imshow(mid_slice_frames[:, :, f], cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
        ax.set_title(f"Frame {f+1}")

    # Hide unused subplots
    for ax in axes[F:]:
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
plot_mid_slice_from_saved_data("/workspaces/PhD/unet3+/data/clean/SEG420/test")

In [ ]:
# the images are grayscale, add a channel dimension
train_images = np.expand_dims(train_images, axis=-1)
val_images = np.expand_dims(val_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

In [ ]:
# Flatten masks for class weight computation
train_masks_flat = train_masks.ravel()
val_masks_flat = val_masks.ravel()
test_masks_flat = test_masks.ravel()

# Unique classes in the training masks (for class weight computation)
classes = np.unique(train_masks_flat)

# Compute class weights
class_weights = class_weight.compute_class_weight('balanced', classes=classes, y=train_masks_flat)
class_weights_dict = {int(classes[i]): class_weights[i] for i in range(len(classes))}
print("Class weights are:", class_weights_dict)


In [ ]:
# Number of classes for segmentation
n_classes = 3 

# Convert masks to one-hot encoding
train_masks_cat = to_categorical(train_masks, num_classes=n_classes)
val_masks_cat = to_categorical(val_masks, num_classes=n_classes)
test_masks_cat = to_categorical(test_masks, num_classes=n_classes)

# Reshape the masks back to add an extra dimension for channels
train_masks_cat = train_masks_cat.reshape((train_masks.shape[0], train_masks.shape[1], train_masks.shape[2], n_classes))
val_masks_cat = val_masks_cat.reshape((val_masks.shape[0], val_masks.shape[1], val_masks.shape[2], n_classes))
test_masks_cat = test_masks_cat.reshape((test_masks.shape[0], test_masks.shape[1], test_masks.shape[2], n_classes))

# Print the shapes to verify
print("Shape of training masks (one-hot):", train_masks_cat.shape)
print("Shape of validation masks (one-hot):", val_masks_cat.shape)
print("Shape of testing masks (one-hot):", test_masks_cat.shape)

In [ ]:
# Check data types and shapesw
print(f"Type of train_images: {type(train_images)}")
print(f"Type of train_masks: {type(train_masks)}")
print(f"Shape of train_images:", train_images.shape)
print(f"Shape of train_masks:", train_masks.shape)

In [ ]:
savedir = '/workspaces/PhD/outputs/SEG420/AugAdd'
os.makedirs(savedir, exist_ok=True)


def plot_pipeline_augmentation_effects_and_save(train_images, train_masks, savedir, num_samples=1):
    """
    Visualize and save the effects of each augmentation in the pipeline.

    Args:
    - train_images: TensorFlow tensor or NumPy array of training images (normalized to [0, 1]).
    - train_masks: TensorFlow tensor or NumPy array of training masks.
    - savedir: Directory to save the augmentation visualizations locally.
    - num_samples: Number of examples to visualize.
    """

    augmentations = {
        "HorizontalFlip": A.HorizontalFlip(p=1.0),
        "RandomRotate90": A.RandomRotate90(p=1.0),
        "RandomBrightnessContrast_Min": A.RandomBrightnessContrast(brightness_limit=(-0.17, -0.0), contrast_limit=(0.0, 0.0), p=1.0),
        "RandomBrightnessContrast_Max": A.RandomBrightnessContrast(brightness_limit=(0.17, 0.0), contrast_limit=(0.0, 0.0), p=1.0),
        "RandomGamma_Min": A.RandomGamma(gamma_limit=(70, 70), p=1.0),
        "RandomGamma_Max": A.RandomGamma(gamma_limit=(120, 120), p=1.0),
        "GaussNoise_Min": A.GaussNoise(var_limit=(30.0,30.0), p=1.0),
        "GaussNoise_Max": A.GaussNoise(var_limit=(100.0, 100.0), p=1.0),
        "CLAHE": A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
        "ShiftScaleRotate_Min": A.ShiftScaleRotate(shift_limit=-0.0625, scale_limit=-0.1, rotate_limit=-10, p=1.0),
        "ShiftScaleRotate_Max": A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=10, p=1.0),
    }


    for sample_idx in range(num_samples):
        # Randomly select an image and its corresponding mask
        idx = random.randint(0, len(train_images) - 1)
        # Directly use NumPy arrays
        original_image = (train_images[idx] * 255).astype(np.uint8)  
        original_mask = train_masks[idx].astype(np.int32)  


        plt.figure(figsize=(15, len(augmentations) * 4))
        plt.subplot(len(augmentations) + 1, 2, 1)
        plt.imshow(original_image, cmap='gray')
        plt.title("Original Image")
        plt.axis('off')

        plt.subplot(len(augmentations) + 1, 2, 2)
        plt.imshow(original_mask, cmap='viridis')
        plt.title("Original Mask")
        plt.axis('off')

        for i, (aug_name, aug) in enumerate(augmentations.items()):
            augmented = aug(image=original_image, mask=original_mask)
            augmented_image = augmented['image'] / 255.0
            augmented_mask = augmented['mask']

            plt.subplot(len(augmentations) + 1, 2, 2 * i + 3)
            plt.imshow(augmented_image.squeeze(), cmap='gray')
            plt.title(f"{aug_name} Image")
            plt.axis('off')

            plt.subplot(len(augmentations) + 1, 2, 2 * i + 4)
            plt.imshow(augmented_mask, cmap='viridis')
            plt.title(f"{aug_name} Mask")
            plt.axis('off')

            plt.tight_layout()

        # Save to local directory
        save_path = os.path.join(savedir, f"augmentation_sample_{sample_idx + 1}.png")
        plt.savefig(save_path)
        print(f"Saved augmentation visualization to {save_path}")


        # Upload to Neptune
        run["Augmentations"].upload(save_path)
        plt.show()

plot_pipeline_augmentation_effects_and_save(train_images, train_masks, savedir=savedir, num_samples=1)

In [ ]:
# Function to visualize and save original and augmented images with masks
def visualize_augmentations(original_image, augmented_image, original_mask, augmented_mask, idx, output_dir):
    """
    Plot original and augmented images with their corresponding masks in a 2x2 grid.
    """
    plt.figure(figsize=(8, 8))

    # Original image
    plt.subplot(2, 2, 1)
    plt.imshow(original_image.squeeze(), cmap='gray')
    plt.title("Original Image")
    plt.axis('off')

    # Augmented image
    plt.subplot(2, 2, 2)
    plt.imshow(augmented_image.squeeze(), cmap='gray')
    plt.title("Augmented Image")
    plt.axis('off')

    # Original mask
    plt.subplot(2, 2, 3)
    plt.imshow(original_mask.squeeze(), cmap='viridis')
    plt.title("Original Mask")
    plt.axis('off')

    # Augmented mask
    plt.subplot(2, 2, 4)
    plt.imshow(augmented_mask.squeeze(), cmap='viridis')
    plt.title("Augmented Mask")
    plt.axis('off')

    plt.tight_layout()

    # Save the plot locally
    save_path = os.path.join(output_dir, f"augmentation_sample_{idx}.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()  # Close the figure to free memory

    # Upload to Neptune
    run[f"Augmentation_Image_Mask/augmented_image_with_mask_{idx}"].upload(File(save_path))



def plot_augmented_test_images_with_masks(train_images, train_masks, train_metadata, augmentation_pipeline, num_samples=5):
    for i in range(num_samples):
        idx = random.randint(0, len(train_images) - 1)

        # Extract metadata
        patient_id, slice_number, frame_number = train_metadata[idx]

        # Prepare image and mask
        image = np.squeeze(train_images[idx])
        image = (image * 255).astype(np.uint8)
        mask = train_masks[idx].astype(np.int32)

        # Apply augmentation
        augmented = augmentation_pipeline(image=image, mask=mask)
        augmented_image = augmented['image'] / 255.0
        augmented_mask = augmented['mask']

        # ✅ Print metadata
        print(f"[{i}] Patient: {patient_id} | Slice: {slice_number} | Frame: {frame_number}")

        # Visualize and save
        visualize_augmentations(
            original_image=train_images[idx],
            augmented_image=augmented_image,
            original_mask=train_masks[idx],
            augmented_mask=augmented_mask,
            idx=i,
            output_dir=savedir
        )


plot_augmented_test_images_with_masks(train_images, train_masks, train_metadata, augmentation_pipeline, num_samples=5)


In [ ]:
# Generate a small batch
sample_gen = data_generator_with_augmentation(train_images, train_masks_cat, batch_size=16)
augmented_images, augmented_masks = next(sample_gen)

# Check dimensions
print("Augmented image shape:", augmented_images.shape) 
print("Augmented mask shape:", augmented_masks.shape)   

 
# Check normalization
check_normalization(augmented_images, "Augmented Train Images")
print("Augmented train images - min pixel value:", np.min(augmented_images))
print("Augmented train images - max pixel value:", np.max(augmented_images))


In [ ]:
# Generate a small batch
sample_gen = data_generator_with_augmentation(train_images, train_masks, batch_size=16)
augmented_images, augmented_masks = next(sample_gen)

# Print unique values in the augmented data
print("Unique values in augmented train masks (after augmentation):", np.unique(augmented_masks))


In [ ]:
# input_shape = [256,256, 1]
# inputs = tf.keras.Input(shape = input_shape)

# BATCH_SIZE = 16
# EPOCHS = 200

# unet3 = unet3plus(inputs, 
#                  rank = 2,  # dimension
#                 #  n_outputs = 3, 
#                  out_channels = n_classes,
#                  add_dropout = 1, # 1 or 0 to add dropout
#                  dropout_rate = 0.3,
#                 ##  base_filters = 32, 
#                  filters = [32, 64, 128, 256, 512],
#                 #  filters = [32, 64, 96, 128, 256],
#                  kernel_size = 3, 
#                 #  stack_num_down = 2,    # 3
#                 #  stack_num_up = 1,   # 3 - 1, 
#                  encoder_block_depth=2,
#                  decoder_block_depth=1,
#                 #  supervision = 1, # 0 or 1 to add supervision 1=True 0=False
#                  deep_supervision=True,
#                  CGM = False) # leave this 0 as it doesn't seem to work 1=True 0=False

# model = tf.keras.Model(inputs = inputs, outputs = unet3.outputs())


# # Define the loss function
# loss = focal_tversky_loss
# # Compile the model using the loss variable
# model.compile(optimizer = tf.keras.optimizers.Adam(), loss = loss, metrics = [dice_coef_no_bkg])


# # Good practice to namespace config
# run['config/loss_function'] = 'focal_tversky_loss'
# run['config/epochs'] = EPOCHS
# run['config/batch_size'] = BATCH_SIZE


# model.summary()

In [ ]:
input_shape = [256,256, 1]
inputs = tf.keras.Input(shape = input_shape)

BATCH_SIZE = 16
EPOCHS = 1

unet3 = unet3plus(inputs, 
                 rank = 2,  # dimension
                 out_channels = n_classes,
                 add_dropout = 1, # 1 or 0 to add dropout
                 dropout_rate = 0.3,
                 filters = [32, 64, 128, 256, 512],
                 kernel_size = 3, 
                 encoder_block_depth=2,
                 decoder_block_depth=1,
                 deep_supervision=True,
                 CGM = False) # leave this 0 as it doesn't seem to work 1=True 0=False

model = tf.keras.Model(inputs = inputs, outputs = unet3.outputs())


model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=focal_tversky_loss,
    metrics=[
        dice_coef_class(1, name='dice_myo'),       # Myocardium Dice
        dice_coef_class(2, name='dice_blood'),     # Blood Pool Dice
        tf.keras.metrics.MeanMetricWrapper(dice_coef_no_bkg, name='dice_no_bkg')  # Mean foreground Dice
    ]
)


# Good practice to namespace config
run['config/loss_function'] = 'focal_tversky_loss'
run['config/epochs'] = EPOCHS
run['config/batch_size'] = BATCH_SIZE


model.summary()

In [ ]:
# class NeptuneLossLogger(Callback):
#     def on_epoch_end(self, epoch, logs=None):
#         if logs is None:
#             return
        
#         for key, value in logs.items():
#             if "loss" in key:
#                 run[f"losses/{key}"].log(value)
#             elif "dice_coef" in key or "metric" in key:
#                 run[f"metrics/{key}"].log(value)

In [ ]:
class NeptuneLossLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        for key, value in logs.items():
            if "loss" in key:
                run[f"losses/{key}"].log(value)
            elif "dice" in key or "accuracy" in key or "metric" in key:
                run[f"metrics/{key}"].log(value * 100 if value < 1.5 else value)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from neptune.integrations.tensorflow_keras import NeptuneCallback


early_stop = EarlyStopping(
    monitor='val_loss',
    mode='min',
    verbose=1,
    patience=20,
    restore_best_weights=True
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        f'models/{model_name}.h5', 
        save_best_only=True, 
        monitor="val_loss", 
        mode="min"
    ),
    NeptuneCallback(run=run),
    NeptuneLossLogger(),
    early_stop
]

In [ ]:

# # Train model
# history = model.fit(
#     train_generator,
#     steps_per_epoch=len(train_images) // BATCH_SIZE,
#     verbose=1,
#     epochs=EPOCHS,
#     callbacks=callbacks,
#     validation_data=(val_images, val_masks_cat),
#     shuffle=False
# )

In [ ]:
# Train on all frames containing segmentation (EDV, ESV, + In-Flow Artifacts)
train_generator = data_generator_with_augmentation(train_images, train_masks_cat, BATCH_SIZE)

# Start timer
start_time = time.time()



# Train model
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_images) // BATCH_SIZE,
    verbose=1,
    epochs=EPOCHS,
    callbacks=callbacks,
    validation_data=(val_images, val_masks_cat),
    shuffle=False
)

# Get the best epoch (based on lowest val_loss)
if 'val_loss' in history.history:
    best_epoch = np.argmin(history.history['val_loss']) + 1  # epochs are 1-indexed
    run["config/training/best_epoch"] = best_epoch
    print(f"Best epoch: {best_epoch}")

# End timer
end_time = time.time()

# Compute elapsed time
elapsed_time = end_time - start_time

# Optional: format nicely as H:M:S
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)

# Print result
print(f"Total training time: {hours}h {minutes}m {seconds}s ({elapsed_time:.2f} seconds)")

# Log to Neptune
run["config/training/total_time_seconds"] = elapsed_time
run["config/training/total_time_hms"] = f"{hours}h {minutes}m {seconds}s"


In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv("training_history.csv", index=False)

# Upload to Neptune
run["config/training_history_csv"].upload("training_history.csv")


In [ ]:
run['model'].upload(f'models/{model_name}.h5')

In [ ]:
# Upload the script to Neptune
run["config/seg_zhi_diceupdated_18.ipynb"].upload("seg_zhi_diceupdated_18.ipynb")

In [ ]:
# Evaluate the model and get the results - UNet3Plus model outputs multiple predictions due to deep supervision
# Selecting the last output (Final Supervision Level)
results = model.evaluate(test_images, test_masks_cat, return_dict=True)

# Print each metric vertically
print("\nMetrics for the Final Supervision Level:")
for key in sorted(results.keys()):
    if 'loss' in key:
        print(f"{key}: {results[key]:.4f}")  
        # Log each loss to Neptune
        run[key].log(results[key])
    elif 'dice_coef_no_bkg' in key:
        print(f"{key}: {results[key] * 100:.2f}%")  
        # Log each Dice coefficient to Neptune
        run[key].log(results[key] * 100)

# The last metric in the results corresponds to the final supervision level's Dice coefficient
final_dice_key = list(results.keys())[-1]  # Get the last key corresponds to the final metric
final_dice_coef = results[final_dice_key]

# Print and log the final Dice coefficient
print("\nFinal Dice Coefficient (for the last supervision level) is = {:.2f}%".format(final_dice_coef * 100))
run['final_dice_coefficient_last_supervision'].log(final_dice_coef * 100)

In [ ]:
# #plot the training and validation accuracy and loss at each epoch
# loss = history.history['loss']
# val_loss = history.history['val_loss']
# epochs = range(1, len(loss) + 1)
# plt.plot(epochs, loss, 'y', label='Training loss')
# plt.plot(epochs, val_loss, 'r', label='Validation loss')
# plt.title('Training and validation loss')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.legend()
# plt.savefig('figure.png', bbox_inches='tight')  

# # Log the figure to Neptune
# run['Loss & Accuracy/Training & Validation Loss (UNet3+, loss=focal_tversky, metrics=dice_coef_no_bkg)'].upload('figure.png')

In [ ]:
# Extract loss values
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)

# Plot
plt.figure(figsize=(7, 5))
plt.plot(epochs, loss, color='orange', label='Training loss')
plt.plot(epochs, val_loss, color='green', label='Validation loss')
# plt.plot(epochs, loss, color='blue', label='Training loss')
# plt.plot(epochs, val_loss, color='green', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('loss_plot.png', bbox_inches='tight')

# Upload to Neptune
run['Loss & Accuracy/Training_Validation_Loss'].upload('loss_plot.png')


In [ ]:
dice_myo = history.history['deepsup_conv_1_2_dice_myo']
val_dice_myo = history.history['val_deepsup_conv_1_2_dice_myo']

dice_blood = history.history['deepsup_conv_1_2_dice_blood']
val_dice_blood = history.history['val_deepsup_conv_1_2_dice_blood']

dice_mean = history.history['deepsup_conv_1_2_dice_no_bkg']
val_dice_mean = history.history['val_deepsup_conv_1_2_dice_no_bkg']


# Prepare epoch range
epochs = range(1, len(dice_myo) + 1)

# Plot both classes' Dice coefficients
plt.figure()
plt.plot(epochs, [d * 100 for d in dice_myo], color='orange',  linestyle='--', label='Train Myocardium Dice')
plt.plot(epochs, [d * 100 for d in val_dice_myo], color='green', linestyle='--', label='Val Myocardium Dice')
plt.plot(epochs, [d * 100 for d in dice_blood], color='orange', label='Train Blood Pool Dice')
plt.plot(epochs, [d * 100 for d in val_dice_blood], color='green', label='Val Blood Pool Dice')

plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Dice (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('dice_combined.png')
run['Loss & Accuracy/Dice_Blood_vs_Myo'].upload('dice_combined.png')


In [ ]:
dice_myo = history.history['deepsup_conv_1_2_dice_myo']
val_dice_myo = history.history['val_deepsup_conv_1_2_dice_myo']

dice_blood = history.history['deepsup_conv_1_2_dice_blood']
val_dice_blood = history.history['val_deepsup_conv_1_2_dice_blood']

dice_mean = history.history['deepsup_conv_1_2_dice_no_bkg']
val_dice_mean = history.history['val_deepsup_conv_1_2_dice_no_bkg']

epochs = range(1, len(dice_myo) + 1)

# Plot myocardium Dice
plt.figure()
plt.plot(epochs, [d * 100 for d in dice_myo], color='orange', label='Train Myocardium Dice')
plt.plot(epochs, [d * 100 for d in val_dice_myo], color='green', label='Val Myocardium Dice')
# plt.title('Dice Coefficient - Myocardium')
plt.title('Training and Validation Accuracy - Myocardium')
plt.xlabel('Epochs')
plt.ylabel('Dice (%)')
plt.legend()
plt.grid(True)
plt.savefig('dice_myo.png')
run['Loss & Accuracy/Dice_Myocardium'].upload('dice_myo.png')

# Plot blood pool Dice
plt.figure()
plt.plot(epochs, [d * 100 for d in dice_blood], color='orange', label='Train Blood Pool Dice')
plt.plot(epochs, [d * 100 for d in val_dice_blood], color='green', label='Val Blood Pool Dice')
# plt.title('Dice Coefficient - Blood Pool')
plt.title('Training and Validation Accuracy - Blood Pool')
plt.xlabel('Epochs')
plt.ylabel('Dice (%)')
plt.legend()
plt.grid(True)
plt.savefig('dice_blood.png')
run['Loss & Accuracy/Dice_Blood_Pool'].upload('dice_blood.png')

# Plot macro-average Dice (no background)
plt.figure()
plt.plot(epochs, [d * 100 for d in dice_mean], color='orange', label='Train Mean Dice (No Bkg)')
plt.plot(epochs, [d * 100 for d in val_dice_mean], color='green', label='Val Mean Dice (No Bkg)')
# plt.title('Macro-Average Dice Coefficient (No Background)')
plt.title('Training and Validation Accuracy - Background')
plt.xlabel('Epochs')
plt.ylabel('Dice (%)')
plt.legend()
plt.grid(True)
plt.savefig('dice_mean.png')
run['Loss & Accuracy/Dice_Mean_NoBkg'].upload('dice_mean.png')


In [ ]:
print(history.history.keys())


In [ ]:
def focal_tversky_loss(y_true, y_pred, alpha=0.5, beta=0.5, gamma=1.0, smooth=1e-6): 
    """
    Focal Tversky loss for multi-class 3D segmentation.

    Args:
    y_true: tensor of shape [B, D, H, W, C]
    y_pred: tensor of shape [B, D, H, W, C]
    alpha: controls the penalty for false positives
    beta: controls the penalty for false negatives
    gamma: focal parameter to down-weight easy examples
    smooth: smoothing constant to avoid division by zero

    Returns:
    loss: computed Focal Tversky loss
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, smooth, 1.0 - smooth)  # Clipping to avoid log(0)
    
    num_classes = 3
    loss = 0.0
    
    for c in range(num_classes):
        y_true_c = y_true[..., c]
        y_pred_c = y_pred[..., c]
        
        true_pos = tf.reduce_sum(y_true_c * y_pred_c)
        false_neg = tf.reduce_sum(y_true_c * (1 - y_pred_c))
        false_pos = tf.reduce_sum((1 - y_true_c) * y_pred_c)
        
        tversky_index = (true_pos + smooth) / (true_pos + alpha * false_neg + beta * false_pos + smooth)
        loss_c = tf.pow((1 - tversky_index), gamma)
        loss += loss_c
    
    loss /= tf.cast(num_classes, tf.float32)  # Averaging over all classes
    return loss







# Dice for a specific class (foreground only)
def dice_coef_class(class_index, name=None, smooth=1e-6):
    """
    Returns a Dice metric for a specific class index (e.g., 1=myocardium, 2=blood pool).
    """
    def dice(y_true, y_pred):
        y_true_c = tf.cast(y_true[..., class_index], tf.float32)
        y_pred_c = tf.clip_by_value(y_pred[..., class_index], 0.0, 1.0)

        y_true_f = tf.reshape(y_true_c, [-1])
        y_pred_f = tf.reshape(y_pred_c, [-1])

        intersection = tf.reduce_sum(y_true_f * y_pred_f)
        union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)
        return (2. * intersection + smooth) / (union + smooth)

    return tf.keras.metrics.MeanMetricWrapper(dice, name=name or f'dice_class_{class_index}')




# Macro-average Dice over all foreground classes
def dice_coef_no_bkg(y_true, y_pred, smooth=1e-6):
    """
    Compute mean Dice Coefficient over all foreground classes (excluding background).
    Assumes multi-class one-hot encoding and softmax predictions.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Remove background (channel 0)
    y_true_fg = y_true[..., 1:]
    y_pred_fg = y_pred[..., 1:]

    # Flatten spatial dimensions, preserve class channels
    y_true_f = tf.reshape(y_true_fg, [-1, tf.shape(y_true_fg)[-1]])
    y_pred_f = tf.reshape(y_pred_fg, [-1, tf.shape(y_pred_fg)[-1]])

    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    denominator = tf.reduce_sum(y_true_f + y_pred_f, axis=0)

    dice = (2. * intersection + smooth) / (denominator + smooth)
    mean_dice = tf.reduce_mean(dice)

    return mean_dice




In [ ]:
# Define the supervision layer (last one)
supervision_layer = 'deepsup_conv_1_2'

# Access the correct metric keys from history
dice_myo = history.history[f'{supervision_layer}_dice_myo']
val_dice_myo = history.history[f'val_{supervision_layer}_dice_myo']
dice_blood = history.history[f'{supervision_layer}_dice_blood']
val_dice_blood = history.history[f'val_{supervision_layer}_dice_blood']
dice_mean = history.history[f'{supervision_layer}_dice_no_bkg']
val_dice_mean = history.history[f'val_{supervision_layer}_dice_no_bkg']

epochs = range(1, len(dice_myo) + 1)


def dice(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.clip_by_value(y_pred, 0.0, 1.0), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)
    return (2. * intersection + smooth) / (union + smooth)



# Reusable metric wrapper
def dice_coef_class(class_index, name=None, smooth=1e-6):
    def wrapped_dice(y_true, y_pred):
        return dice(y_true[..., class_index], y_pred[..., class_index], smooth)
    return tf.keras.metrics.MeanMetricWrapper(wrapped_dice, name=name or f'dice_class_{class_index}')



# Correctly define the custom objects using actual functions
custom_objects = {
    'focal_tversky_loss': focal_tversky_loss,
    'dice_coef_no_bkg': dice_coef_no_bkg,
    'ResizeAndConcatenate': ResizeAndConcatenate,
    'dice_myo': dice_coef_class(1, name='dice_myo'),
    'dice_blood': dice_coef_class(2, name='dice_blood'),
    'dice': dice  
}



# Load the trained model
model = tf.keras.models.load_model('/workspaces/PhD/models/17_zhi.h5', custom_objects=custom_objects)
print('✅ Loaded model')

# Display the input & output shapes
print("✅ Model loaded successfully")
print(f"Model input shape: {model.input_shape}")
print(f"Model output shape: {model.output_shape}")


In [ ]:
# Define the test folder path
test_folder = "/workspaces/PhD/unet3+/data/clean/SEG420/test"


# Define the predictions directory
predictions_dir = "/workspaces/PhD/unet3+/data/clean/SEG420/test_predictions"
os.makedirs(test_folder, exist_ok=True)
os.makedirs(predictions_dir, exist_ok=True)  


# List all patient directories in the test folder
patient_dirs = [os.path.join(test_folder, patient) for patient in os.listdir(test_folder) if os.path.isdir(os.path.join(test_folder, patient))]

for patient_dir in patient_dirs:
    patient_id = os.path.basename(patient_dir)
    
    # Load images and masks for the patient
    images_path = os.path.join(patient_dir, "images.npy")
    if not os.path.exists(images_path):
        print(f"Images not found for patient {patient_id}")
        continue
    
    # Load images
    patient_images = np.load(images_path)  # Shape: (H, W, S, F)
    H, W, S, F = patient_images.shape
    print(f"Loaded data for patient {patient_id} with shape {patient_images.shape}")
    
    # Initialize predictions array
    predictions = np.zeros_like(patient_images, dtype=np.uint8)  # Shape: (H, W, S, F)

    # Process data in batches for prediction
    for frame_idx in range(F):
        frame_images = patient_images[:, :, :, frame_idx]  # Shape: (H, W, S)
        
        # Add batch and channel dimensions: (H, W, S) -> (S, H, W, 1)
        frame_images = np.moveaxis(frame_images, -1, 0)[..., np.newaxis]  # Shape: (S, H, W, 1)
       
        # Make predictions
        frame_predictions = model.predict(frame_images, verbose=1)  # Predict for all slices in the frame
        # Print the shape of the raw predictions
        frame_predictions = frame_predictions[-1]  # Use the last output if the model has multiple outputs
        print(f"Shape of raw frame_predictions: {frame_predictions.shape}")
        # Convert predictions to class labels
        frame_predictions = np.argmax(frame_predictions, axis=-1).astype(np.uint8)  # Shape: (S, H, W)
        # Print the shape after applying argmax
        print(f"Shape of frame_predictions after argmax: {frame_predictions.shape}")

        # Reshape back to (H, W, S) and store in the predictions array
        predictions[:, :, :, frame_idx] = np.moveaxis(frame_predictions, 0, -1)

    # Save the predictions for the patient
    patient_predictions_dir = os.path.join(predictions_dir, patient_id)
    os.makedirs(patient_predictions_dir, exist_ok=True)
    np.save(os.path.join(patient_predictions_dir, "predictions.npy"), predictions)
    
    print(f"Predictions saved for patient {patient_id} in {patient_predictions_dir}")


In [ ]:
# Get sorted list of patient directories
patient_dirs = sorted([d for d in os.listdir(test_folder) if os.path.isdir(os.path.join(test_folder, d))])

# Select patient and load data
patient_index = 0  # Select the first test patient
slice_idx = 5  # Select the slice index

patient_id = patient_dirs[patient_index]
patient_dir = os.path.join(test_folder, patient_id)
predictions_file = os.path.join(predictions_dir, patient_id, "predictions.npy")
images_file = os.path.join(patient_dir, "images.npy")
masks_file = os.path.join(patient_dir, "masks.npy")


# Load data
images = np.load(images_file)  # Shape: (H, W, S, F)
masks = np.load(masks_file)    # Shape: (H, W, S, F)
predictions = np.load(predictions_file)  # Shape: (H, W, S, F)

# Plot images and overlays for both frames
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for row_idx, (frame_idx, frame_name) in enumerate(zip([0, 1], ["Diastolic", "Systolic"])):
    image = images[:, :, slice_idx, frame_idx]
    mask = masks[:, :, slice_idx, frame_idx]
    prediction = predictions[:, :, slice_idx, frame_idx]

    # Original image
    axes[row_idx, 0].imshow(image, cmap="gray")
    axes[row_idx, 0].set_title(f"DICOM ({frame_name})")
    axes[row_idx, 0].axis("off")
    
    # Ground truth mask
    axes[row_idx, 1].imshow(mask, cmap="viridis")
    axes[row_idx, 1].set_title(f"Ground Truth Mask ({frame_name})")
    axes[row_idx, 1].axis("off")
    
    # Predicted mask
    axes[row_idx, 2].imshow(prediction, cmap="viridis")
    axes[row_idx, 2].set_title(f"Predicted Mask ({frame_name})")
    axes[row_idx, 2].axis("off")
    
    # Overlay of predicted mask on the image
    axes[row_idx, 3].imshow(image, cmap="gray")
    axes[row_idx, 3].imshow(prediction, cmap="viridis", alpha=0.2)  
    axes[row_idx, 3].set_title(f"Prediction Overlay ({frame_name})")
    axes[row_idx, 3].axis("off")

plt.tight_layout()
plt.show()



In [ ]:
print(f"Shapes for Patient {patient_id}:")
print(f"  Images: {images.shape}")
print(f"  Masks: {masks.shape}")
print(f"  Predictions: {predictions.shape}")


In [ ]:
# Before island removal
# Dice Score Calculation using SciPy & Plotting GIFs 

# Class mappings
class_mapping = {"myocardium": 1, "blood_pool": 2}  # Class IDs for myocardium and blood pool
results = {}

# Function to compute Dice similarity using SciPy
def compute_dice_similarity(y_true, y_pred):
    """
    Compute Dice similarity using SciPy's dice dissimilarity.
    Dice similarity = 1 - Dice dissimilarity.
    """
    return 1 - distance.dice(y_true.flatten(), y_pred.flatten())


# Function to calculate Dice scores for a single patient
def calculate_dice_scores(masks, predictions, patient_id):
    """
    Calculate Dice scores for diastolic and systolic phases and compute averages for diastolic, systolic, and all.
    """
    dice_scores = {}
    diastolic_scores = []
    systolic_scores = []
    
    for frame_idx, frame_name in zip([0, 1], ["diastolic", "systolic"]):
        for class_name, class_id in class_mapping.items():
            # Create binary masks for the current class
            y_true = (masks[..., frame_idx] == class_id).astype(bool)
            y_pred = (predictions[..., frame_idx] == class_id).astype(bool)
            
            # Compute Dice similarity
            dice_similarity = compute_dice_similarity(y_true, y_pred)
            dice_scores[f"{frame_name}_{class_name}"] = dice_similarity
            
            # Append to respective lists for later averaging
            if frame_name == "diastolic":
                diastolic_scores.append(dice_similarity)
            else:
                systolic_scores.append(dice_similarity)
    
    # Compute averages
    dice_scores["average_diastolic"] = np.mean(diastolic_scores)
    dice_scores["average_systolic"] = np.mean(systolic_scores)
    dice_scores["average"] = np.mean(list(dice_scores.values()))

    # Print Dice scores in the desired format for the patient
    print(f"\nPatient: {patient_id}")
    for key, value in dice_scores.items():
        print(f"  {key}: {value:.4f}")

    return dice_scores




# Function to generate GIF with Dice Scores
def gif_animation_for_patient_with_scores_singlephase(
    patient_images, patient_metadata, dice_scores, patient_id, output_dir
):
    fig, axarr = plt.subplots(2, 2, figsize=(10, 10))

    # Retrieve diastolic and systolic frames for the given patient ID
    dia_frame, sys_frame = 0, 1  # Diastolic (0) and systolic (1) frames are fixed

    # Separate out the diastolic and systolic images based on their frames
    diastolic_images = [img for img, meta in zip(patient_images, patient_metadata) if meta[2] == dia_frame]
    systolic_images = [img for img, meta in zip(patient_images, patient_metadata) if meta[2] == sys_frame]

    def update(frame_index):
        # Clear previous frames
        for ax in axarr.flatten():
            ax.clear()

        # Extract image components for the diastolic and systolic frames
        original_image_dia, ground_truth_mask_dia, predicted_mask_dia = diastolic_images[frame_index]
        original_image_sys, ground_truth_mask_sys, predicted_mask_sys = systolic_images[frame_index]

        # Display images with dynamic alpha masking for transparency using class_mapping ori= 0.8
        axarr[0, 0].imshow(original_image_dia, cmap='gray')
        axarr[0, 0].imshow(ground_truth_mask_dia == class_mapping["myocardium"], 
                           alpha=(ground_truth_mask_dia == class_mapping["myocardium"]) * 0.7, cmap='Blues')  # Myocardium
        axarr[0, 0].imshow(ground_truth_mask_dia == class_mapping["blood_pool"], 
                           alpha=(ground_truth_mask_dia == class_mapping["blood_pool"]) * 0.7, cmap='jet')   # Blood pool
        axarr[0, 0].set_title('GT Diastole', fontsize=10)
        axarr[0, 0].axis('off')

        axarr[0, 1].imshow(original_image_dia, cmap='gray')
        axarr[0, 1].imshow(predicted_mask_dia == class_mapping["myocardium"], 
                           alpha=(predicted_mask_dia == class_mapping["myocardium"]) * 0.7, cmap='Blues')  # Myocardium
        axarr[0, 1].imshow(predicted_mask_dia == class_mapping["blood_pool"], 
                           alpha=(predicted_mask_dia == class_mapping["blood_pool"]) * 0.7, cmap='jet')    # Blood pool
        axarr[0, 1].set_title('Prediction Diastole', fontsize=10)
        axarr[0, 1].axis('off')

        axarr[1, 0].imshow(original_image_sys, cmap='gray')
        axarr[1, 0].imshow(ground_truth_mask_sys == class_mapping["myocardium"], 
                           alpha=(ground_truth_mask_sys == class_mapping["myocardium"]) * 0.7, cmap='Blues')  # Myocardium
        axarr[1, 0].imshow(ground_truth_mask_sys == class_mapping["blood_pool"], 
                           alpha=(ground_truth_mask_sys == class_mapping["blood_pool"]) * 0.7, cmap='jet')   # Blood pool
        axarr[1, 0].set_title('GT Systole', fontsize=10)
        axarr[1, 0].axis('off')

        axarr[1, 1].imshow(original_image_sys, cmap='gray')
        axarr[1, 1].imshow(predicted_mask_sys == class_mapping["myocardium"], 
                           alpha=(predicted_mask_sys == class_mapping["myocardium"]) * 0.7, cmap='Blues')  # Myocardium
        axarr[1, 1].imshow(predicted_mask_sys == class_mapping["blood_pool"], 
                           alpha=(predicted_mask_sys == class_mapping["blood_pool"]) * 0.7, cmap='jet')    # Blood pool
        axarr[1, 1].set_title('Prediction Systole', fontsize=10)
        axarr[1, 1].axis('off')

        # Update the figure's main title with the test patient ID
        fig.suptitle(f'Test Patient ID: {patient_id}', fontsize=12)

        # Fetch dice scores or use placeholders if not available
        diastolic_myo_dice = dice_scores.get(f"diastolic_myocardium", "N/A")
        diastolic_blood_dice = dice_scores.get(f"diastolic_blood_pool", "N/A")
        systolic_myo_dice = dice_scores.get(f"systolic_myocardium", "N/A")
        systolic_blood_dice = dice_scores.get(f"systolic_blood_pool", "N/A")

        # Add Dice scores as subtitles
        fig.text(
            0.5, 0.91,
            f'Diastole: Myo Dice {diastolic_myo_dice:.4f}, Blood Dice {diastolic_blood_dice:.4f}',
            ha='center', va='center', fontsize=10
        )

        fig.text(
            0.5, 0.49,
            f'Systole: Myo Dice {systolic_myo_dice:.4f}, Blood Dice {systolic_blood_dice:.4f}',
            ha='center', va='center', fontsize=10
        )

    plt.subplots_adjust(wspace=0.01, hspace=0.2)

    anim = animation.FuncAnimation(fig, update, frames=len(diastolic_images), interval=700)

    # Save the animation to the output directory
    output_path = os.path.join(output_dir, f"{patient_id}_with_dice.gif")
    anim.save(output_path, writer='pillow')
    plt.close(fig)

    return output_path





# Load and save predictions
for patient_id in sorted(os.listdir(test_folder)):
    patient_test_folder = os.path.join(test_folder, patient_id)
    predictions_file = os.path.join(predictions_dir, patient_id, "predictions.npy")
    images_file = os.path.join(patient_test_folder, "images.npy")
    masks_file = os.path.join(patient_test_folder, "masks.npy")

    # Check if predictions file exists
    if os.path.exists(predictions_file):
        print(f"Predictions already exist for patient {patient_id}. Skipping.")
        continue

    print(f"Processing Patient {patient_id}")

    # Ensure the patient folder exists
    if not all([os.path.exists(images_file), os.path.exists(masks_file)]):
        print(f"Missing data for patient {patient_id}. Skipping.")
        continue

    # Load the images
    images = np.load(images_file)  # Shape: (H, W, S, F)
    masks = np.load(masks_file)  # Shape: (H, W, S, F)
    # Initialize the predictions array
    predictions = np.zeros_like(images, dtype=np.uint8)

    # Process each frame for predictions
    for frame_idx in range(images.shape[-1]):
        frame_images = images[..., frame_idx]  # Select the frame
        # Prepare data for prediction
        frame_images = np.moveaxis(frame_images, -1, 0)[..., np.newaxis]

        # Perform prediction using the model
        frame_predictions = model.predict(frame_images, verbose=1)
        frame_predictions = np.argmax(frame_predictions[-1], axis=-1).astype(np.uint8)

        # Reshape predictions back and store them
        predictions[..., frame_idx] = np.moveaxis(frame_predictions, 0, -1)

    # Save predictions
    os.makedirs(os.path.dirname(predictions_file), exist_ok=True)
    np.save(predictions_file, predictions)
    print(f"Saved predictions for patient {patient_id} to {predictions_file}")



# Dir for Dice Score & GIF Generation
# output_dir = "/workspaces/PhD/outputs/SEG292_ALLframes/GIFs"
output_dir = "/workspaces/PhD/outputs/SEG420/Test_GIFs_EDV_ESV"
os.makedirs(output_dir, exist_ok=True) # Ensure the directory exists

# Calculate Dice scores
all_dice_scores = []
for patient_id in sorted(os.listdir(test_folder)):
    patient_dir = os.path.join(test_folder, patient_id)
    images_file = os.path.join(patient_dir, "images.npy")
    masks_file = os.path.join(patient_dir, "masks.npy")
    predictions_file = os.path.join(predictions_dir, patient_id, "predictions.npy")
    
    # Load patient data
    if not all([os.path.exists(images_file), os.path.exists(masks_file), os.path.exists(predictions_file)]):
        print(f"Skipping {patient_id} due to missing data")
        continue
    
    images = np.load(images_file)
    masks = np.load(masks_file)
    predictions = np.load(predictions_file)


    # Calculate Dice scores
    dice_scores = calculate_dice_scores(masks, predictions, patient_id)

    # Store in results dictionary
    results[patient_id] = dice_scores

    # Add patient_id and scores to the list
    patient_scores = {"Patient ID": patient_id}
    patient_scores.update(dice_scores)          # Add all the Dice scores
    all_dice_scores.append(patient_scores)



    # Prepare images and metadata
    patient_images = []
    patient_metadata = []
    for slice_idx in range(images.shape[2]):  # Iterate over slices
        for frame_idx in [0, 1]:  # 0: Diastolic, 1: Systolic
            original_image = images[:, :, slice_idx, frame_idx]
            ground_truth_mask = masks[:, :, slice_idx, frame_idx]
            predicted_mask = predictions[:, :, slice_idx, frame_idx]
            patient_images.append((original_image, ground_truth_mask, predicted_mask))
            patient_metadata.append((patient_id, slice_idx, frame_idx))
    
    # Create and save GIF
    gif_path = gif_animation_for_patient_with_scores_singlephase(
        patient_images, patient_metadata, dice_scores, patient_id, output_dir
    )
    print(f"Generated GIF for Patient ID {patient_id}: {gif_path}")
    run[f"GIFs Before Island Removal/{patient_id}"].upload(gif_path)

    

# Save Dice scores to CSV and Excel
# output_dir = "/workspaces/PhD/outputs/SEG292_ALLframes/Dice Scores"
output_dir = "/workspaces/PhD/outputs/SEG420/Test_DiceScore_EDV_ESV"
os.makedirs(output_dir, exist_ok=True)  

# Save Dice Scores to DataFrame
dice_scores_df = pd.DataFrame(all_dice_scores)
sorted_results = dice_scores_df.sort_values("average", ascending=False)

# Calculate diastolic and systolic overall statistics
diastolic_scores = dice_scores_df[["diastolic_myocardium", "diastolic_blood_pool"]].mean(axis=1)
systolic_scores = dice_scores_df[["systolic_myocardium", "systolic_blood_pool"]].mean(axis=1)



# Define file paths - Save CSV and Excel
csv_file = os.path.join(output_dir, "SciPy_dice_scores.csv")
excel_file = os.path.join(output_dir, "SciPy_dice_scores.xlsx")
# Save to CSV
dice_scores_df.to_csv(csv_file, index=False)
# Save to Excel
dice_scores_df.to_excel(excel_file, index=False)
print(f"Saved Dice scores to:\n  CSV: {csv_file}\n  Excel: {excel_file}")




# Save Summary File
summary_file = os.path.join(output_dir, "SciPy_dice_scores_summary.txt")
with open(summary_file, "w") as f:
    # Write patient-level scores
    f.write("Patient Dice Scores (SciPy):\n")
    f.write("=====================\n")
    for _, row in sorted_results.iterrows():
        f.write(f"Patient: {row['Patient ID']}\n")
        f.write(f"  diastolic_myocardium: {row['diastolic_myocardium']:.4f}\n")
        f.write(f"  diastolic_blood_pool: {row['diastolic_blood_pool']:.4f}\n")
        f.write(f"  systolic_myocardium: {row['systolic_myocardium']:.4f}\n")
        f.write(f"  systolic_blood_pool: {row['systolic_blood_pool']:.4f}\n")
        f.write(f"  average_diastolic: {row['average_diastolic']:.4f}\n")
        f.write(f"  average_systolic: {row['average_systolic']:.4f}\n")
        f.write(f"  average: {row['average']:.4f}\n")
        f.write("\n")



    # Write overall statistics
    f.write("\nOverall Statistics:\n")
    f.write("====================\n")

    # Define helper function to extract and write stats
    def write_stats(column_name, display_name):
        best_idx = dice_scores_df[column_name].idxmax()
        best_score = dice_scores_df[column_name].max()
        best_patient = dice_scores_df.loc[best_idx, 'Patient ID']

        sorted_indices = dice_scores_df[column_name].sort_values().index
        num_patients = len(dice_scores_df[column_name])
        if num_patients % 2 == 1:  # Odd
            median_idx = sorted_indices[num_patients // 2]
            median_score = dice_scores_df[column_name].loc[median_idx]
            median_patient = dice_scores_df.loc[median_idx, 'Patient ID']
        else:  # Even
            lower_idx = sorted_indices[num_patients // 2 - 1]
            upper_idx = sorted_indices[num_patients // 2]
            median_score = (dice_scores_df[column_name].loc[lower_idx] + dice_scores_df[column_name].loc[upper_idx]) / 2
            median_patient = f"{dice_scores_df.loc[lower_idx, 'Patient ID']} and {dice_scores_df.loc[upper_idx, 'Patient ID']}"

        lowest_idx = dice_scores_df[column_name].idxmin()
        lowest_score = dice_scores_df[column_name].min()
        lowest_patient = dice_scores_df.loc[lowest_idx, 'Patient ID']

        f.write(f"Best {display_name} Dice Score: {best_score:.4f} (Patient ID: {best_patient})\n")
        f.write(f"Median {display_name} Dice Score: {median_score:.4f} (Patient ID: {median_patient})\n")
        f.write(f"Lowest {display_name} Dice Score: {lowest_score:.4f} (Patient ID: {lowest_patient})\n")
        f.write("-------------------------------\n")

    # Write stats for each specific category
    write_stats("diastolic_myocardium", "diastolic_myocardium")
    write_stats("diastolic_blood_pool", "diastolic_blood_pool")
    write_stats("systolic_myocardium", "systolic_myocardium")
    write_stats("systolic_blood_pool", "systolic_blood_pool")

    # Write combined blood pool stats
    blood_pool_scores = dice_scores_df[['diastolic_blood_pool', 'systolic_blood_pool']].mean(axis=1)
    best_blood_pool_idx = blood_pool_scores.idxmax()
    best_blood_pool_score = blood_pool_scores.max()
    best_blood_pool_patient = dice_scores_df.loc[best_blood_pool_idx, 'Patient ID']

    sorted_blood_pool_indices = blood_pool_scores.sort_values().index
    if len(blood_pool_scores) % 2 == 1:  # Odd
        median_blood_pool_idx = sorted_blood_pool_indices[len(blood_pool_scores) // 2]
        median_blood_pool_score = blood_pool_scores.loc[median_blood_pool_idx]
        median_blood_pool_patient = dice_scores_df.loc[median_blood_pool_idx, 'Patient ID']
    else:  # Even
        lower_idx = sorted_blood_pool_indices[len(blood_pool_scores) // 2 - 1]
        upper_idx = sorted_blood_pool_indices[len(blood_pool_scores) // 2]
        median_blood_pool_score = (blood_pool_scores.loc[lower_idx] + blood_pool_scores.loc[upper_idx]) / 2
        median_blood_pool_patient = f"{dice_scores_df.loc[lower_idx, 'Patient ID']} and {dice_scores_df.loc[upper_idx, 'Patient ID']}"

    lowest_blood_pool_idx = blood_pool_scores.idxmin()
    lowest_blood_pool_score = blood_pool_scores.min()
    lowest_blood_pool_patient = dice_scores_df.loc[lowest_blood_pool_idx, 'Patient ID']

    f.write(f"Best Blood Pool Dice Score: {best_blood_pool_score:.4f} (Patient ID: {best_blood_pool_patient})\n")
    f.write(f"Median Blood Pool Dice Score: {median_blood_pool_score:.4f} (Patient ID: {median_blood_pool_patient})\n")
    f.write(f"Lowest Blood Pool Dice Score: {lowest_blood_pool_score:.4f} (Patient ID: {lowest_blood_pool_patient})\n")
    f.write("-------------------------------\n")

    # Write combined myocardium stats
    myocardium_scores = dice_scores_df[['diastolic_myocardium', 'systolic_myocardium']].mean(axis=1)
    best_myocardium_idx = myocardium_scores.idxmax()
    best_myocardium_score = myocardium_scores.max()
    best_myocardium_patient = dice_scores_df.loc[best_myocardium_idx, 'Patient ID']

    sorted_myocardium_indices = myocardium_scores.sort_values().index
    if len(myocardium_scores) % 2 == 1:  # Odd
        median_myocardium_idx = sorted_myocardium_indices[len(myocardium_scores) // 2]
        median_myocardium_score = myocardium_scores.loc[median_myocardium_idx]
        median_myocardium_patient = dice_scores_df.loc[median_myocardium_idx, 'Patient ID']
    else:  # Even
        lower_idx = sorted_myocardium_indices[len(myocardium_scores) // 2 - 1]
        upper_idx = sorted_myocardium_indices[len(myocardium_scores) // 2]
        median_myocardium_score = (myocardium_scores.loc[lower_idx] + myocardium_scores.loc[upper_idx]) / 2
        median_myocardium_patient = f"{dice_scores_df.loc[lower_idx, 'Patient ID']} and {dice_scores_df.loc[upper_idx, 'Patient ID']}"

    lowest_myocardium_idx = myocardium_scores.idxmin()
    lowest_myocardium_score = myocardium_scores.min()
    lowest_myocardium_patient = dice_scores_df.loc[lowest_myocardium_idx, 'Patient ID']

    f.write(f"Best Myocardium Dice Score: {best_myocardium_score:.4f} (Patient ID: {best_myocardium_patient})\n")
    f.write(f"Median Myocardium Dice Score: {median_myocardium_score:.4f} (Patient ID: {median_myocardium_patient})\n")
    f.write(f"Lowest Myocardium Dice Score: {lowest_myocardium_score:.4f} (Patient ID: {lowest_myocardium_patient})\n")
    f.write("\n")



    # Write sorted patient scores (Highest to Lowest)
    f.write("Patient Average Dice Scores (Highest To Lowest):\n")
    f.write("===================================\n")
    for _, row in sorted_results.iterrows():
        f.write(f"Patient {row['Patient ID']}: {row['average']:.4f}\n")

print(f"Saved Dice Scores Summary to {summary_file}")


# Upload summary text file to Neptune
run["Dice Score/SciPy_dice_scores_summary.txt"].upload(summary_file)

# Upload CSV and Excel files to Neptune
run["Dice Score/SciPy_dice_scores.csv"].upload(csv_file)
run["Dice Score/SciPy_dice_scores.xlsx"].upload(excel_file)


In [ ]:
from skimage.measure import label, regionprops
import numpy as np

def remove_inconsistent_slices(
    segmentation_slices,
    min_slices=2,
    min_area=10,
    distance_threshold=40
):
    """
    Removes inconsistent segmentations robustly:
    - Keeps any component that:
        - Spans at least min_slices slices, AND
        - Is within distance_threshold of the main centroid
    - Ensures large far-away blobs are removed.

    Args:
        segmentation_slices: (H, W, S) array
        min_slices: min number of slices to keep a component
        min_area: min voxel count to keep a component
        distance_threshold: max centroid distance to keep

    Returns:
        cleaned_segmentation: (H, W, S) array
    """
    H, W, S = segmentation_slices.shape
    cleaned_segmentation = np.zeros_like(segmentation_slices, dtype=np.uint8)

    for class_label, label_name in [(1, "myocardium"), (2, "blood_pool")]:
        binary_mask = (segmentation_slices == class_label).astype(np.uint8)
        labels_3d = label(binary_mask, connectivity=1)

        # Collect all candidate components
        regions = [r for r in regionprops(labels_3d) if r.area >= min_area]
        if not regions:
            continue

        # Find the component with the most slices (dominant region)
        dominant_region = max(
            regions,
            key=lambda r: len(set(c[2] for c in r.coords))
        )
        dominant_centroid = np.array(dominant_region.centroid)

        print(f"\nClass {label_name}: dominant region centroid: {dominant_centroid}")

        # Now process each component
        for region in regions:
            slices_present = set(c[2] for c in region.coords)
            centroid = np.array(region.centroid)
            distance = np.linalg.norm(centroid - dominant_centroid)

            print(f"Component {region.label}:")
            print(f"  Slices: {sorted(slices_present)}")
            print(f"  Num slices: {len(slices_present)}")
            print(f"  Area: {region.area}")
            print(f"  Centroid: {centroid}")
            print(f"  Distance to dominant: {distance:.2f}")

            if len(slices_present) >= min_slices and distance <= distance_threshold:
                # Keep it
                for c in region.coords:
                    cleaned_segmentation[c[0], c[1], c[2]] = class_label
            else:
                print("  --> Discarded")

        print("Unique labels in cleaned_segmentation after processing this class:", np.unique(cleaned_segmentation))

    return cleaned_segmentation






# Directories for output
cleaned_predictions_dir = "/workspaces/PhD/outputs/SEG420/After_Islands_Removal_npy"
output_gif_dir = "/workspaces/PhD/outputs/SEG420/GIFs_After_Island_Removal"
cleaned_output_dir = "/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal"

os.makedirs(cleaned_predictions_dir, exist_ok=True)
os.makedirs(output_gif_dir, exist_ok=True)
os.makedirs(cleaned_output_dir, exist_ok=True)

results = {}

# Iterate through all patients in the test folder
for patient_id in sorted(os.listdir(test_folder)):
    print(f"\nProcessing Patient {patient_id}")

    patient_test_folder = os.path.join(test_folder, patient_id)
    predictions_file = os.path.join(predictions_dir, patient_id, "predictions.npy")
    cleaned_predictions_path = os.path.join(cleaned_predictions_dir, f"{patient_id}_cleaned_predictions.npy")
    images_file = os.path.join(patient_test_folder, "images.npy")
    masks_file = os.path.join(patient_test_folder, "masks.npy")

    if not os.path.exists(predictions_file):
        print(f"Predictions not found for patient {patient_id}. Skipping.")
        continue

    print(f"Loading predictions for Patient {patient_id}")
    predictions = np.load(predictions_file)  # Shape: (H, W, S, F)
    H, W, S, F = predictions.shape

    cleaned_predictions = np.zeros_like(predictions, dtype=np.uint8)

    for f in range(F):
        volume = predictions[:, :, :, f]
        print(f"\nFrame {f}: unique labels BEFORE cleaning:", np.unique(volume))

        cleaned_volume = remove_inconsistent_slices(volume)

        print(f"Frame {f}: unique labels AFTER cleaning:", np.unique(cleaned_volume))

        cleaned_predictions[:, :, :, f] = cleaned_volume

    # Final check for all frames
    print("Unique labels in cleaned_predictions AFTER processing all frames:", np.unique(cleaned_predictions))

    np.save(cleaned_predictions_path, cleaned_predictions)
    print(f"✅ Cleaned predictions saved to {cleaned_predictions_path}")

    if not (os.path.exists(images_file) and os.path.exists(masks_file)):
        print(f"Missing images or masks for patient {patient_id}. Skipping.")
        continue

    images = np.load(images_file)
    masks = np.load(masks_file)

    dice_scores = calculate_dice_scores(masks, cleaned_predictions, patient_id)
    results[patient_id] = dice_scores

    patient_images = []
    patient_metadata = []
    for slice_idx in range(images.shape[2]):
        for frame_idx in [0, 1]:
            original_image = images[:, :, slice_idx, frame_idx]
            ground_truth_mask = masks[:, :, slice_idx, frame_idx]
            predicted_mask = cleaned_predictions[:, :, slice_idx, frame_idx]
            patient_images.append((original_image, ground_truth_mask, predicted_mask))
            patient_metadata.append((patient_id, slice_idx, frame_idx))

    gif_path = gif_animation_for_patient_with_scores_singlephase(
        patient_images, patient_metadata, dice_scores, patient_id, output_gif_dir
    )
    print(f"✅ Generated GIF for Patient {patient_id}: {gif_path}")

    run[f"GIFs After Island Removal/{patient_id}"].upload(gif_path)
    print(f"✅ Uploaded GIF for Patient {patient_id} to Neptune.\n")



In [ ]:
# Create per-patient Dice DataFrame
all_dice_scores = []
for patient_id, dice_scores in results.items():
    all_dice_scores.append({
        "Patient ID": patient_id,
        "diastolic_myocardium": dice_scores.get("diastolic_myocardium", 0),
        "diastolic_blood_pool": dice_scores.get("diastolic_blood_pool", 0),
        "systolic_myocardium": dice_scores.get("systolic_myocardium", 0),
        "systolic_blood_pool": dice_scores.get("systolic_blood_pool", 0),
        "average_diastolic": dice_scores.get("average_diastolic", 0),
        "average_systolic": dice_scores.get("average_systolic", 0),
        "average": dice_scores.get("average", 0),
    })

cleaned_dice_scores_df = pd.DataFrame(all_dice_scores)
cleaned_sorted_results = cleaned_dice_scores_df.sort_values("average", ascending=False)

# Compute extra averages
cleaned_dice_scores_df["average_blood_pool"] = (
    cleaned_dice_scores_df["diastolic_blood_pool"] + cleaned_dice_scores_df["systolic_blood_pool"]
) / 2
cleaned_dice_scores_df["average_myocardium"] = (
    cleaned_dice_scores_df["diastolic_myocardium"] + cleaned_dice_scores_df["systolic_myocardium"]
) / 2

# Save CSV/Excel
cleaned_csv_file = os.path.join(cleaned_output_dir, "SciPy_dice_scores_cleaned.csv")
cleaned_excel_file = os.path.join(cleaned_output_dir, "SciPy_dice_scores_cleaned.xlsx")
cleaned_dice_scores_df.to_csv(cleaned_csv_file, index=False)
cleaned_dice_scores_df.to_excel(cleaned_excel_file, index=False)
print(f"✅ Saved per-patient Dice scores to:\n  {cleaned_csv_file}\n  {cleaned_excel_file}")

# Per-frame DataFrame
per_frame_rows = []
for _, row in cleaned_dice_scores_df.iterrows():
    per_frame_rows.append({
        "Patient_ID": row["Patient ID"],
        "Frame_Type": "ED",
        "Average_Dice": row["average_diastolic"],
        "Myocardium_Dice": row["diastolic_myocardium"],
        "BloodPool_Dice": row["diastolic_blood_pool"]
    })
    per_frame_rows.append({
        "Patient_ID": row["Patient ID"],
        "Frame_Type": "ES",
        "Average_Dice": row["average_systolic"],
        "Myocardium_Dice": row["systolic_myocardium"],
        "BloodPool_Dice": row["systolic_blood_pool"]
    })

per_frame_df = pd.DataFrame(per_frame_rows).sort_values(by="Average_Dice", ascending=False).reset_index(drop=True)
per_frame_csv_file = os.path.join(cleaned_output_dir, "SciPy_dice_scores_per_frame.csv")
per_frame_df.to_csv(per_frame_csv_file, index=False)
print(f"✅ Saved per-frame Dice scores to: {per_frame_csv_file}")

# Identify best/median/worst frames
best_frame = per_frame_df.iloc[0]
median_frame = per_frame_df.iloc[len(per_frame_df)//2]
worst_frame = per_frame_df.iloc[-1]


# Define function for Block A stats
def write_cleaned_stats(f, df, column_name, display_name):
    best_idx = df[column_name].idxmax()
    best_score = df[column_name].max()
    best_patient = df.loc[best_idx, "Patient ID"]

    sorted_indices = df[column_name].sort_values().index
    num_patients = len(df[column_name])
    if num_patients % 2 == 1:
        median_idx = sorted_indices[num_patients // 2]
        median_score = df[column_name].loc[median_idx]
        median_patient = df.loc[median_idx, "Patient ID"]
    else:
        lower_idx = sorted_indices[num_patients // 2 - 1]
        upper_idx = sorted_indices[num_patients // 2]
        median_score = (
            df[column_name].loc[lower_idx] + df[column_name].loc[upper_idx]
        ) / 2
        median_patient = (
            f"{df.loc[lower_idx, 'Patient ID']} and {df.loc[upper_idx, 'Patient ID']}"
        )

    lowest_idx = df[column_name].idxmin()
    lowest_score = df[column_name].min()
    lowest_patient = df.loc[lowest_idx, "Patient ID"]

    f.write(f"Best {display_name} Dice Score: {best_score:.2f} (Patient ID: {best_patient})\n")
    f.write(f"Median {display_name} Dice Score: {median_score:.2f} (Patient ID: {median_patient})\n")
    f.write(f"Lowest {display_name} Dice Score: {lowest_score:.2f} (Patient ID: {lowest_patient})\n")
    f.write("-------------------------------\n")

# Identify best/median/worst frames
best_frame = per_frame_df.iloc[0]
median_frame = per_frame_df.iloc[len(per_frame_df)//2]
worst_frame = per_frame_df.iloc[-1]

# Write everything to summary
summary_file = os.path.join(cleaned_output_dir, "SciPy_dice_scores_summary_cleaned.txt")
with open(summary_file, "w") as f:
    # Block A per-patient detailed listing
    f.write("Patient Dice Scores (SciPy, After Island Removal):\n")
    f.write("==================================================\n")
    for _, row in cleaned_sorted_results.iterrows():
        f.write(f"Patient: {row['Patient ID']}\n")
        f.write(f"  diastolic_myocardium: {row['diastolic_myocardium']:.2f}\n")
        f.write(f"  diastolic_blood_pool: {row['diastolic_blood_pool']:.2f}\n")
        f.write(f"  systolic_myocardium: {row['systolic_myocardium']:.2f}\n")
        f.write(f"  systolic_blood_pool: {row['systolic_blood_pool']:.2f}\n")
        f.write(f"  average_diastolic: {row['average_diastolic']:.2f}\n")
        f.write(f"  average_systolic: {row['average_systolic']:.2f}\n")
        f.write(f"  average: {row['average']:.2f}\n")
        f.write("\n")

    # Block A stats
    f.write("Overall Statistics After Island Removal:\n")
    f.write("=========================================\n")
    write_cleaned_stats(f, cleaned_dice_scores_df, "diastolic_myocardium", "diastolic_myocardium")
    write_cleaned_stats(f, cleaned_dice_scores_df, "diastolic_blood_pool", "diastolic_blood_pool")
    write_cleaned_stats(f, cleaned_dice_scores_df, "systolic_myocardium", "systolic_myocardium")
    write_cleaned_stats(f, cleaned_dice_scores_df, "systolic_blood_pool", "systolic_blood_pool")

    # Block A per-patient ranking
    f.write("\nPatient Average Dice Scores (Highest to Lowest, After Island Removal):\n")
    f.write("=======================================================================\n")
    for _, row in cleaned_sorted_results.iterrows():
        f.write(f"Patient {row['Patient ID']}: {row['average']:.2f}\n")

    f.write("=============================================\n")
    write_cleaned_stats(f, cleaned_dice_scores_df, "average", "Average (All Regions)")
    write_cleaned_stats(f, cleaned_dice_scores_df, "average_blood_pool", "Average Blood Pool")
    write_cleaned_stats(f, cleaned_dice_scores_df, "average_myocardium", "Average Myocardium")

    # Per-frame scores
    f.write("\nPer-frame Dice Scores (ED and ES separately):\n")
    f.write("=============================================\n")
    for _, row in per_frame_df.iterrows():
        f.write(f"Patient: {row['Patient_ID']}  Frame: {row['Frame_Type']}\n")
        f.write(f"  Average_Dice: {row['Average_Dice']:.2f}\n")
        f.write(f"  Myocardium_Dice: {row['Myocardium_Dice']:.2f}\n")
        f.write(f"  BloodPool_Dice: {row['BloodPool_Dice']:.2f}\n")
        f.write("-------------------------------\n")

    # Best/Median/Worst frames
    f.write("\nBest, Median, Worst Frames (Ranked by Average_Dice):\n")
    f.write("====================================================\n")
    f.write(f"Best Frame:\n")
    f.write(f"  Patient: {best_frame['Patient_ID']}\n")
    f.write(f"  Frame Type: {best_frame['Frame_Type']}\n")
    f.write(f"  Average Dice: {best_frame['Average_Dice']:.2f}\n\n")
    f.write(f"Median Frame:\n")
    f.write(f"  Patient: {median_frame['Patient_ID']}\n")
    f.write(f"  Frame Type: {median_frame['Frame_Type']}\n")
    f.write(f"  Average Dice: {median_frame['Average_Dice']:.2f}\n\n")
    f.write(f"Worst Frame:\n")
    f.write(f"  Patient: {worst_frame['Patient_ID']}\n")
    f.write(f"  Frame Type: {worst_frame['Frame_Type']}\n")
    f.write(f"  Average Dice: {worst_frame['Average_Dice']:.2f}\n\n")

print(f"✅ Saved summary file to {summary_file}")

# Upload to Neptune
run["Dice Score/After_Island_Removal_SciPy_dice_scores.csv"].upload(cleaned_csv_file)
run["Dice Score/After_Island_Removal_SciPy_dice_scores_.xlsx"].upload(cleaned_excel_file)
run["Dice Score/After_Island_Removal_SciPy_dice_scores_per_frame.csv"].upload(per_frame_csv_file)
run["Dice Score/After_Island_Removal_SciPy_dice_scores_summary.txt"].upload(summary_file)
print("✅ All Dice score files uploaded to Neptune.")


In [ ]:
After island removal dice mean ± sd for test ds

In [ ]:
import os
import sys
import io
import pandas as pd
from statsmodels.stats.anova import AnovaRM
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
from tabulate import tabulate
import neptune

# ----------------------------
# ✅ Setup Paths and Logging
# ----------------------------
save_dir = "/workspaces/PhD/outputs/SEG420/Dice_Score/Stats"
os.makedirs(save_dir, exist_ok=True)
log_path = os.path.join(save_dir, "Dice_Stats_Log.txt")

# Capture printed output
log_buffer = io.StringIO()
original_stdout = sys.stdout
sys.stdout = log_buffer

# ----------------------------
# ✅ Start of Script Logic
# ----------------------------

# Load CSVs
per_frame_df = pd.read_csv("/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal/SciPy_dice_scores_per_frame.csv")
per_patient_df = pd.read_csv("/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal/SciPy_dice_scores_cleaned.csv")
per_patient_df = per_patient_df.rename(columns={"Patient ID": "Patient_ID"})

# Precompute per-patient averages
per_patient_df["average_blood_pool"] = (per_patient_df["diastolic_blood_pool"] + per_patient_df["systolic_blood_pool"]) / 2
per_patient_df["average_myocardium"] = (per_patient_df["diastolic_myocardium"] + per_patient_df["systolic_myocardium"]) / 2

# Subsets
df_ed = per_frame_df[per_frame_df["Frame_Type"] == "ED"]
df_es = per_frame_df[per_frame_df["Frame_Type"] == "ES"]
metrics = ["Average_Dice", "Myocardium_Dice", "BloodPool_Dice"]

# Print summary
print("\n==============================")
print("PER-PATIENT Overall Averages (ED+ES averaged per patient)")
print(f"  Blood Pool:   {per_patient_df['average_blood_pool'].mean():.2f} ± {per_patient_df['average_blood_pool'].std():.2f}")
print(f"  Myocardium:   {per_patient_df['average_myocardium'].mean():.2f} ± {per_patient_df['average_myocardium'].std():.2f}")

print("\nPER-VOLUME Dice Scores (ED only)")
for metric in metrics:
    print(f"  {metric}: {df_ed[metric].mean():.2f} ± {df_ed[metric].std():.2f}")

print("\nPER-VOLUME Dice Scores (ES only)")
for metric in metrics:
    print(f"  {metric}: {df_es[metric].mean():.2f} ± {df_es[metric].std():.2f}")

print("\nPER-VOLUME Dice Scores (ED+ES pooled) ✅")
for metric in metrics:
    print(f"  {metric}: {per_frame_df[metric].mean():.2f} ± {per_frame_df[metric].std():.2f}")
print("==============================\n")

# Prepare for ANOVA
df_long = pd.melt(
    per_patient_df,
    id_vars=["Patient_ID"],
    value_vars=[
        "diastolic_myocardium", "diastolic_blood_pool",
        "systolic_myocardium", "systolic_blood_pool"
    ],
    var_name="Phase_Structure",
    value_name="Dice_Score"
)
df_long["Phase"] = df_long["Phase_Structure"].apply(lambda x: "Diastolic" if "diastolic" in x else "Systolic")
df_long["Structure"] = df_long["Phase_Structure"].apply(lambda x: "Myocardium" if "myocardium" in x else "Blood Pool")

# Mean ± SD
means = df_long.groupby(["Phase", "Structure"])["Dice_Score"].agg(["mean", "std"]).reset_index()
desired_order = [
    ("Blood Pool", "Diastolic"),
    ("Blood Pool", "Systolic"),
    ("Myocardium", "Diastolic"),
    ("Myocardium", "Systolic")
]
means["SortKey"] = means.apply(lambda row: desired_order.index((row["Structure"], row["Phase"])), axis=1)
means = means.sort_values("SortKey").drop("SortKey", axis=1).reset_index(drop=True)

print("\n✅ Mean ± SD Dice Scores per Phase & Structure:")
for _, row in means.iterrows():
    print(f"- {row['Structure']} {row['Phase']}: {row['mean']:.2f} ± {row['std']:.2f}")

means_file = os.path.join(save_dir, "Mean_Dice_Scores_per_Group.csv")
means.to_csv(means_file, index=False)

# ANOVA
anova = AnovaRM(df_long, depvar="Dice_Score", subject="Patient_ID", within=["Phase", "Structure"]).fit()
anova_table = anova.anova_table.reset_index().rename(columns={"index": "Effect", "Pr > F": "p-value"})

print("\n✅ Repeated Measures ANOVA Results:")
print(tabulate(anova_table, headers="keys", tablefmt="grid", floatfmt=".3f", showindex=False))

anova_file = os.path.join(save_dir, "ANOVA_summary.txt")
with open(anova_file, "w") as f:
    f.write(tabulate(anova_table, headers="keys", tablefmt="grid", floatfmt=".3f"))

# Paired t-tests
pivot = df_long.pivot(index="Patient_ID", columns=["Structure", "Phase"], values="Dice_Score")
comparisons = [
    (("Myocardium", "Diastolic"), ("Myocardium", "Systolic")),
    (("Blood Pool", "Diastolic"), ("Blood Pool", "Systolic")),
    (("Myocardium", "Diastolic"), ("Blood Pool", "Diastolic")),
    (("Myocardium", "Systolic"), ("Blood Pool", "Systolic")),
]

results = []
for ((s1, p1), (s2, p2)) in comparisons:
    t_stat, p_val = ttest_rel(pivot[(s1, p1)], pivot[(s2, p2)])
    mean_diff = (pivot[(s1, p1)] - pivot[(s2, p2)]).mean()
    results.append({
        "Comparison": f"{s1} ({p1}) vs {s2} ({p2})",
        "Mean_Difference": mean_diff,
        "t_stat": t_stat,
        "raw_p_value": p_val
    })

# FDR correction
p_values = [r["raw_p_value"] for r in results]
reject, pvals_corrected, _, _ = multipletests(p_values, alpha=0.05, method="fdr_bh")
for i, r in enumerate(results):
    r["FDR_corrected_p"] = pvals_corrected[i]
    r["Significant"] = "Yes" if reject[i] else "No"

# Print t-test results
print("\n✅ Targeted Post-Hoc Paired t-tests (FDR-corrected):")
print(tabulate(pd.DataFrame(results), headers="keys", tablefmt="grid", floatfmt=".4f", showindex=range(1, len(results)+1)))

interaction_p = anova_table.loc[anova_table["Effect"] == "Phase:Structure", "p-value"].values[0]
print("\nNote: A significant interaction between phase and structure was detected (p = {:.4f}).".format(interaction_p))

ttest_file = os.path.join(save_dir, "Targeted_Pairwise_ttests.csv")
pd.DataFrame(results).to_csv(ttest_file, index=False)

print("\n✅ All results saved to CSV and text files.")

# ----------------------------
# ✅ END of Script Logic
# ----------------------------

# Restore stdout and save log
sys.stdout = original_stdout
with open(log_path, "w") as f:
    f.write(log_buffer.getvalue())

# Print everything captured back into notebook
print(log_buffer.getvalue())

# Upload to Neptune
run["Dice Score/Stats_Files/Mean_Dice_Scores_per_Group.csv"].upload(means_file)
run["Dice Score/Stats_Files/ANOVA_summary.txt"].upload(anova_file)
run["Dice Score/Stats_Files/Targeted_Pairwise_ttests.csv"].upload(ttest_file)
run["Dice Score/Stats_Files/Dice_Stats_Log.txt"].upload(log_path)

print("📤 All outputs uploaded to Neptune ✅")


In [ ]:
✅ select best / median / worst individual frames (ED or ES) based on their average Dice score, instead of averaging across ED and ES...

In [ ]:
# PNG & mp4.. 

def upload_combined_grid_like_paper_figure_equal_spacing_best_median_worst_per_frame(
    per_frame_df,
    test_folder,
    predictions_folder,
    run
):
    """
    Create a Figure 2-like grid showing the best, median, and worst frames across all patients,
    regardless of ED or ES.
    """
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    import numpy as np
    import os
    import math
    from neptune.types import File

    # Sort per-frame DataFrame
    sorted_frames = per_frame_df.sort_values("Average_Dice", ascending=False).reset_index(drop=True)

    # Select Best, Median, Worst frames
    best_frame = sorted_frames.iloc[0]
    median_frame = sorted_frames.iloc[len(sorted_frames) // 2]
    worst_frame = sorted_frames.iloc[-1]

    selected_frames = [
        ("Best", best_frame),
        ("Median", median_frame),
        ("Worst", worst_frame)
    ]

    fixed_n_cols = 3
    max_n_rows = 0

    # Compute max number of rows needed among all selected cases
    for case_name, row in selected_frames:
        patient_id = row["Patient_ID"]
        images = np.load(os.path.join(test_folder, patient_id, "images.npy"))
        S = images.shape[2]
        n_rows = math.ceil(S / fixed_n_cols)
        max_n_rows = max(max_n_rows, n_rows)

    fig_width = 11.5
    fig_height = 4.5 * len(selected_frames) + max_n_rows * 2.5
    fig = plt.figure(figsize=(fig_width, fig_height), dpi=400)

    outer_grid = gridspec.GridSpec(
        len(selected_frames), 2,
        figure=fig,
        hspace=0.3,
        wspace=0.3
    )

    for row_idx, (case_name, row) in enumerate(selected_frames):
        patient_id = row["Patient_ID"]
        frame_type = row["Frame_Type"]
        frame_number = 0 if frame_type == "ED" else 1
        dice_score = row["Average_Dice"]

        print(f"\nSelected [{case_name}] — Patient [{patient_id}] — Frame [{frame_type}] — Dice: {dice_score:.2f}")
    

        images = np.load(os.path.join(test_folder, patient_id, "images.npy"))
        masks = np.load(os.path.join(test_folder, patient_id, "masks.npy"))
        predictions = np.load(os.path.join(predictions_folder, f"{patient_id}_cleaned_predictions.npy"))

        H, W, S, F = images.shape
        n_rows = max_n_rows

        left_grid = gridspec.GridSpecFromSubplotSpec(
            n_rows, fixed_n_cols,
            subplot_spec=outer_grid[row_idx, 0],
            hspace=0.09,
            wspace=0.00
        )
        right_grid = gridspec.GridSpecFromSubplotSpec(
            n_rows, fixed_n_cols,
            subplot_spec=outer_grid[row_idx, 1],
            hspace=0.09,
            wspace=0.00
        )

        left_bbox = outer_grid[row_idx, 0].get_position(fig)
        right_bbox = outer_grid[row_idx, 1].get_position(fig)

        full_width_center = (left_bbox.x0 + right_bbox.x1) / 2
        title_y = max(left_bbox.y1, right_bbox.y1) + 0.02

        fig.text(
            full_width_center,
            title_y,
            # f"{case_name} Case (Dice = {dice_score:.2f}) — Frame: {frame_type}",
            f"{case_name} Case (Dice = {dice_score:.2f})",
            ha='center',
            va='bottom',
            fontsize=22,
            fontweight='bold'
        )

        subtitle_y = max(left_bbox.y1, right_bbox.y1) + 0.01

        fig.text(
            (left_bbox.x0 + left_bbox.x1) / 2, subtitle_y,
            "Ground Truth",
            ha='center', va='top', fontsize=16, fontweight='semibold'
        )
        fig.text(
            (right_bbox.x0 + right_bbox.x1) / 2, subtitle_y,
            "DL Segmentation",
            ha='center', va='top', fontsize=16, fontweight='semibold'
        )

        alpha_value = 0.5

        # Ground Truth
        for slice_idx in range(S):
            dicom = images[:, :, slice_idx, frame_number]
            gt_mask = masks[:, :, slice_idx, frame_number]
            dicom_norm = (dicom - dicom.min()) / (dicom.max() - dicom.min())

            row_pos = slice_idx // fixed_n_cols
            col_pos = slice_idx % fixed_n_cols

            ax = fig.add_subplot(left_grid[row_pos, col_pos])
            ax.imshow(dicom_norm, cmap='gray')
            ax.imshow(gt_mask == 2, alpha=(gt_mask == 2) * alpha_value, cmap='jet')
            ax.imshow(gt_mask == 1, alpha=(gt_mask == 1) * alpha_value, cmap='Blues')
            ax.axis("off")

        # Predictions
        for slice_idx in range(S):
            dicom = images[:, :, slice_idx, frame_number]
            pred_mask = predictions[:, :, slice_idx, frame_number]
            dicom_norm = (dicom - dicom.min()) / (dicom.max() - dicom.min())

            row_pos = slice_idx // fixed_n_cols
            col_pos = slice_idx % fixed_n_cols

            ax = fig.add_subplot(right_grid[row_pos, col_pos])
            ax.imshow(dicom_norm, cmap='gray')
            ax.imshow(pred_mask == 2, alpha=(pred_mask == 2) * alpha_value, cmap='jet')
            ax.imshow(pred_mask == 1, alpha=(pred_mask == 1) * alpha_value, cmap='Blues')
            ax.axis("off")

    plt.tight_layout()

    neptune_folder = "Figures_Updated/Figure2_Best_Median_Worst_PerFrame2"
    run[neptune_folder].upload(File.as_image(fig))
    print(f"\n✅ PAPER FIGURE (Best/Median/Worst per-frame) uploaded to Neptune at [{neptune_folder}]")

    plt.close(fig)
    


upload_combined_grid_like_paper_figure_equal_spacing_best_median_worst_per_frame(
    per_frame_df=per_frame_df,
    test_folder="/workspaces/PhD/unet3+/data/clean/SEG420/test",
    predictions_folder="/workspaces/PhD/outputs/SEG420/After_Islands_Removal_npy",
    run=run
)



In [ ]:
mp4

In [ ]:
def generate_best_median_worst_mp4_per_frame(
    per_frame_df,
    test_folder,
    predictions_folder,
    run,
    output_dir="/workspaces/PhD/outputs/SEG420/MP4"
):
    """
    Generates 3 MP4 videos showing the best, median, and worst frames across all patients and frames.
    """
    import matplotlib.pyplot as plt
    import matplotlib.animation as animation
    import matplotlib.gridspec as gridspec
    import numpy as np
    import os
    from neptune.types import File

    os.makedirs(output_dir, exist_ok=True)

    # Sort per-frame DataFrame
    sorted_df = per_frame_df.sort_values("Average_Dice", ascending=False).reset_index(drop=True)

    best_frame = sorted_df.iloc[0]
    median_frame = sorted_df.iloc[len(sorted_df) // 2]
    worst_frame = sorted_df.iloc[-1]

    selected_frames = [
        ("Best", best_frame),
        ("Median", median_frame),
        ("Worst", worst_frame)
    ]

    for case_name, row in selected_frames:
        patient_id = row["Patient_ID"]
        frame_type = row["Frame_Type"]
        frame_number = 0 if frame_type == "ED" else 1
        dice_score = row["Average_Dice"]

        # Load data
        images = np.load(os.path.join(test_folder, patient_id, "images.npy"))
        masks = np.load(os.path.join(test_folder, patient_id, "masks.npy"))
        predictions = np.load(os.path.join(predictions_folder, f"{patient_id}_cleaned_predictions.npy"))

        H, W, S, F = images.shape

        # Prepare figure
        fig = plt.figure(figsize=(10, 6), dpi=200)
        gs = gridspec.GridSpec(2, 2, height_ratios=[0.1, 1], hspace=0.1, wspace=0.05)
        ax_title = fig.add_subplot(gs[0, :])
        ax_left = fig.add_subplot(gs[1, 0])
        ax_right = fig.add_subplot(gs[1, 1])

        alpha_value = 0.5

        def update(slice_idx):
            ax_left.clear()
            ax_right.clear()
            ax_title.clear()

            dicom = images[:, :, slice_idx, frame_number]
            gt_mask = masks[:, :, slice_idx, frame_number]
            pred_mask = predictions[:, :, slice_idx, frame_number]

            dicom_norm = (dicom - dicom.min()) / (dicom.max() - dicom.min())

            # Title
            ax_title.text(
                0.5,
                0.5,
                f"{case_name} Case (Dice = {dice_score:.2f}) - Slice {slice_idx+1}/{S}",
                ha="center",
                va="center",
                fontsize=14,
                fontweight="bold"
            )
            ax_title.axis("off")

 

            # Ground Truth
            ax_left.imshow(dicom_norm, cmap="gray")
            ax_left.imshow(gt_mask == 2, alpha=(gt_mask == 2) * alpha_value, cmap="jet")
            ax_left.imshow(gt_mask == 1, alpha=(gt_mask == 1) * alpha_value, cmap="Blues")
            ax_left.set_title("Ground Truth", fontsize=12)
            ax_left.axis("off")

            # Prediction
            ax_right.imshow(dicom_norm, cmap="gray")
            ax_right.imshow(pred_mask == 2, alpha=(pred_mask == 2) * alpha_value, cmap="jet")
            ax_right.imshow(pred_mask == 1, alpha=(pred_mask == 1) * alpha_value, cmap="Blues")
            ax_right.set_title("DL Segmentation", fontsize=12)
            ax_right.axis("off")

        ani = animation.FuncAnimation(
            fig, update,
            frames=range(S),
            interval=700,
            repeat_delay=1000
        )

        mp4_path = os.path.join(output_dir, f"{case_name}_Frame_{patient_id}_{frame_type}.mp4")
        ani.save(mp4_path, writer="ffmpeg", dpi=200)
        plt.close(fig)

        print(f"✅ Saved {case_name} MP4 to {mp4_path}")

        # Upload to Neptune
        neptune_folder = "MP4_BestMedianWorst_PerFrame"
        run[f"{neptune_folder}/{case_name}_Frame_{patient_id}_{frame_type}.mp4"].upload(File(mp4_path))

        print(f"✅ Uploaded {case_name} MP4 to Neptune under {neptune_folder}")



generate_best_median_worst_mp4_per_frame(
    per_frame_df=per_frame_df,
    test_folder="/workspaces/PhD/unet3+/data/clean/SEG420/test",
    predictions_folder="/workspaces/PhD/outputs/SEG420/After_Islands_Removal_npy",
    run=run
)



In [ ]:
Correlation and BA 

In [ ]:
# CARDIAC VOLUME CAL AFTER ISLANDS REMOVAL


# Constants for pixel size and slice thickness
pixel_spacing = 0.1  # in mm
slice_thickness = 1.0  # in mm

# Function to calculate volume from segmentation mask in mm³
def calculate_volume_from_mask(mask, pixel_spacing, slice_thickness):
    pixel_area = pixel_spacing ** 2  # in mm²
    blood_pool_area = np.sum(mask == 2)  # Count pixels where mask == 2 (blood pool)
    total_area = blood_pool_area * pixel_area  # in mm²
    volume = total_area * slice_thickness  # in mm³
    return volume


def ensure_edv_esv(edv, esv, patient_id, label):
    if esv > edv:
        print(f"Warning: ESV > EDV for patient {patient_id} ({label}). Swapping values.")
        return esv, edv  # Swap the values if ESV is larger than EDV
    return edv, esv



# List all patient directories in the test folder
patient_dirs = [os.path.join(test_folder, patient) for patient in os.listdir(test_folder) if os.path.isdir(os.path.join(test_folder, patient))]

# Initialize dictionaries for volume calculations
edv_pred_dict, esv_pred_dict, sv_pred_dict, ef_pred_dict = {}, {}, {}, {}
edv_gt_dict, esv_gt_dict, sv_gt_dict, ef_gt_dict = {}, {}, {}, {}
frame_numbers_dict = {}  # <--- NEW: store EDV/ESV frame numbers per patient

# Process all test patients
for patient_dir in patient_dirs:
    patient_id = os.path.basename(patient_dir)
    
    # Load test images, ground truth masks, and cleaned predictions
    images_path = os.path.join(patient_dir, "images.npy")
    masks_path = os.path.join(patient_dir, "masks.npy")
    cleaned_predictions_path = os.path.join(cleaned_predictions_dir, f"{patient_id}_cleaned_predictions.npy")

    if not (os.path.exists(images_path) and os.path.exists(masks_path) and os.path.exists(cleaned_predictions_path)):
        print(f"Missing data for patient {patient_id}")
        continue

    images = np.load(images_path)
    masks = np.load(masks_path)
    predictions = np.load(cleaned_predictions_path)
    
    # Get real frame numbers from metadata
    real_frame_numbers = sorted(set(meta[2] for meta in test_metadata if meta[0] == patient_id))
    if len(real_frame_numbers) < 2:
        print(f"Warning: Patient {patient_id} has fewer than 2 frames. Skipping.")
        continue

    # Use first two frames as EDV and ESV
    edv_frame, esv_frame = real_frame_numbers[0], real_frame_numbers[1]
    frame_numbers_dict[patient_id] = (edv_frame, esv_frame)  # <--- save per-patient

    # Process EDV and ESV frames for cardiac volume calculations
    for slice_idx in range(images.shape[2]):
        for frame_idx, frame_name in zip([0, 1], ["EDV", "ESV"]):  # index in arrays
            if frame_name == "EDV":
                diastolic_mask_pred = predictions[:, :, slice_idx, frame_idx]
                diastolic_mask_gt = masks[:, :, slice_idx, frame_idx]
                diastolic_volume_pred = calculate_volume_from_mask(diastolic_mask_pred, pixel_spacing, slice_thickness)
                edv_pred_dict[patient_id] = edv_pred_dict.get(patient_id, 0) + diastolic_volume_pred
                diastolic_volume_gt = calculate_volume_from_mask(diastolic_mask_gt, pixel_spacing, slice_thickness)
                edv_gt_dict[patient_id] = edv_gt_dict.get(patient_id, 0) + diastolic_volume_gt
            elif frame_name == "ESV":
                systolic_mask_pred = predictions[:, :, slice_idx, frame_idx]
                systolic_mask_gt = masks[:, :, slice_idx, frame_idx]
                systolic_volume_pred = calculate_volume_from_mask(systolic_mask_pred, pixel_spacing, slice_thickness)
                esv_pred_dict[patient_id] = esv_pred_dict.get(patient_id, 0) + systolic_volume_pred
                systolic_volume_gt = calculate_volume_from_mask(systolic_mask_gt, pixel_spacing, slice_thickness)
                esv_gt_dict[patient_id] = esv_gt_dict.get(patient_id, 0) + systolic_volume_gt

# Calculate Stroke Volume (SV) and Ejection Fraction (EF)
for patient_id in edv_pred_dict.keys():
    edv_pred, esv_pred = ensure_edv_esv(edv_pred_dict[patient_id], esv_pred_dict[patient_id], patient_id, "Predicted")
    edv_gt, esv_gt = ensure_edv_esv(edv_gt_dict[patient_id], esv_gt_dict[patient_id], patient_id, "Ground Truth")
    
    sv_pred = edv_pred - esv_pred
    ef_pred = (sv_pred / edv_pred) * 100
    sv_gt = edv_gt - esv_gt
    ef_gt = (sv_gt / edv_gt) * 100

    sv_pred_dict[patient_id] = sv_pred
    ef_pred_dict[patient_id] = ef_pred
    sv_gt_dict[patient_id] = sv_gt
    ef_gt_dict[patient_id] = ef_gt

    # Print per-patient results
    edv_frame, esv_frame = frame_numbers_dict[patient_id]
    print(f"Patient ID: {patient_id}")
    print(f"  EDV Frame: {edv_frame}")
    print(f"  ESV Frame: {esv_frame}")
    print(f"  Ground Truth EDV: {edv_gt:.2f} µL, Predicted EDV: {edv_pred:.2f} µL")
    print(f"  Ground Truth ESV: {esv_gt:.2f} µL, Predicted ESV: {esv_pred:.2f} µL")
    print(f"  Ground Truth SV: {sv_gt:.2f} µL, Predicted SV: {sv_pred:.2f} µL")
    print(f"  Ground Truth EF: {ef_gt:.2f} %, Predicted EF: {ef_pred:.2f} %\n")

# Calculate differences
edv_diff = {pid: abs(edv_gt_dict[pid] - edv_pred_dict[pid]) for pid in edv_gt_dict}
esv_diff = {pid: abs(esv_gt_dict[pid] - esv_pred_dict[pid]) for pid in esv_gt_dict}
sv_diff = {pid: abs(sv_gt_dict[pid] - sv_pred_dict[pid]) for pid in sv_gt_dict}
ef_diff = {pid: abs(ef_gt_dict[pid] - ef_pred_dict[pid]) for pid in ef_gt_dict}

# Sort and top 5
sorted_edv_diff = sorted(edv_diff.items(), key=lambda x: x[1], reverse=True)[:5]
sorted_esv_diff = sorted(esv_diff.items(), key=lambda x: x[1], reverse=True)[:5]
sorted_sv_diff = sorted(sv_diff.items(), key=lambda x: x[1], reverse=True)[:5]
sorted_ef_diff = sorted(ef_diff.items(), key=lambda x: x[1], reverse=True)[:5]

# Print top 5
print("\nTop 5 patients with most differences in EDV:")
for pid, diff in sorted_edv_diff:
    print(f"Patient ID: {pid}, Difference: {diff:.2f} µL")

print("\nTop 5 patients with most differences in ESV:")
for pid, diff in sorted_esv_diff:
    print(f"Patient ID: {pid}, Difference: {diff:.2f} µL")

print("\nTop 5 patients with most differences in SV:")
for pid, diff in sorted_sv_diff:
    print(f"Patient ID: {pid}, Difference: {diff:.2f} µL")

print("\nTop 5 patients with most differences in EF:")
for pid, diff in sorted_ef_diff:
    print(f"Patient ID: {pid}, Difference: {diff:.2f} %")

# Save to CSV/Excel
volume_df = pd.DataFrame({
    "Patient_ID": list(edv_gt_dict.keys()),
    "EDV_GT_µL": [edv_gt_dict[pid] for pid in edv_gt_dict],
    "EDV_Pred_µL": [edv_pred_dict[pid] for pid in edv_pred_dict],
    "ESV_GT_µL": [esv_gt_dict[pid] for pid in esv_gt_dict],
    "ESV_Pred_µL": [esv_pred_dict[pid] for pid in esv_pred_dict],
    "SV_GT_µL": [sv_gt_dict[pid] for pid in sv_gt_dict],
    "SV_Pred_µL": [sv_pred_dict[pid] for pid in sv_pred_dict],
    "EF_GT_percent": [ef_gt_dict[pid] for pid in ef_gt_dict],
    "EF_Pred_percent": [ef_pred_dict[pid] for pid in ef_pred_dict],
})

volumes_csv_file = os.path.join(cleaned_output_dir, "Cardiac_Volumes_After_Island_Removal.csv")
volumes_excel_file = os.path.join(cleaned_output_dir, "Cardiac_Volumes_After_Island_Removal.xlsx")

volume_df.to_csv(volumes_csv_file, index=False)
volume_df.to_excel(volumes_excel_file, index=False)

# Upload to Neptune
run["Stats/Cardiac_Volumes_After_Island_Removal.csv"].upload(volumes_csv_file)
run["Stats/Cardiac_Volumes_After_Island_Removal.xlsx"].upload(volumes_excel_file)

print(f"\n✅ Cardiac volumes saved to:\n  CSV: {volumes_csv_file}\n  Excel: {volumes_excel_file}")
print("✅ Cardiac volumes CSV/Excel uploaded to Neptune.")



In [ ]:
heart
Ground truth - second observer

In [ ]:
import os
import numpy as np
import pandas as pd
import datetime
from scipy.stats import shapiro, ttest_rel, wilcoxon, spearmanr
from pingouin import intraclass_corr
from tabulate import tabulate

# Constants
pixel_spacing = 0.1
slice_thickness = 1.0
myocardium_density = 1.05

# Helper functions
def calculate_volume_from_mask(mask, pixel_spacing, slice_thickness):
    pixel_area = pixel_spacing ** 2
    blood_pool_area = np.sum(mask == 2)
    return blood_pool_area * pixel_area * slice_thickness

def calculate_myocardium_mass(mask, pixel_spacing, slice_thickness, density):
    pixel_area = pixel_spacing ** 2
    myocardium_area = np.sum(mask == 1)
    myocardium_volume = myocardium_area * pixel_area * slice_thickness
    return myocardium_volume * density

def ensure_edv_esv(edv, esv, patient_id, label):
    if esv > edv:
        print(f"Warning: ESV > EDV for patient {patient_id} ({label}). Swapping.")
        return esv, edv
    return edv, esv

# Prepare patient directories
patient_dirs = [
    os.path.join(test_folder, p)
    for p in os.listdir(test_folder)
    if os.path.isdir(os.path.join(test_folder, p))
]

# Initialize dicts
edv_gt_dict, edv_pred_dict = {}, {}
esv_gt_dict, esv_pred_dict = {}, {}
sv_gt_dict, sv_pred_dict = {}, {}
ef_gt_dict, ef_pred_dict = {}, {}
myocardium_mass_gt_ed, myocardium_mass_pred_ed = {}, {}
myocardium_mass_gt_es, myocardium_mass_pred_es = {}, {}

# Process each patient
for patient_dir in patient_dirs:
    patient_id = os.path.basename(patient_dir)
    images_path = os.path.join(patient_dir, "images.npy")
    masks_path = os.path.join(patient_dir, "masks.npy")
    preds_path = os.path.join(cleaned_predictions_dir, f"{patient_id}_cleaned_predictions.npy")

    if not (os.path.exists(images_path) and os.path.exists(masks_path) and os.path.exists(preds_path)):
        print(f"Missing data for {patient_id}")
        continue

    images = np.load(images_path)
    masks = np.load(masks_path)
    preds = np.load(preds_path)

    edv_gt, esv_gt, edv_pred, esv_pred = 0,0,0,0

    for slice_idx in range(images.shape[2]):
        m_edv = masks[:,:,slice_idx,0]
        m_esv = masks[:,:,slice_idx,1]
        p_edv = preds[:,:,slice_idx,0]
        p_esv = preds[:,:,slice_idx,1]

        edv_gt += calculate_volume_from_mask(m_edv, pixel_spacing, slice_thickness)
        edv_pred += calculate_volume_from_mask(p_edv, pixel_spacing, slice_thickness)
        esv_gt += calculate_volume_from_mask(m_esv, pixel_spacing, slice_thickness)
        esv_pred += calculate_volume_from_mask(p_esv, pixel_spacing, slice_thickness)

    edv_gt, esv_gt = ensure_edv_esv(edv_gt, esv_gt, patient_id, "GT")
    edv_pred, esv_pred = ensure_edv_esv(edv_pred, esv_pred, patient_id, "Pred")

    sv_gt = edv_gt - esv_gt
    sv_pred = edv_pred - esv_pred
    ef_gt = (sv_gt / edv_gt) * 100
    ef_pred = (sv_pred / edv_pred) * 100

    edv_gt_dict[patient_id] = edv_gt
    esv_gt_dict[patient_id] = esv_gt
    sv_gt_dict[patient_id] = sv_gt
    ef_gt_dict[patient_id] = ef_gt
    edv_pred_dict[patient_id] = edv_pred
    esv_pred_dict[patient_id] = esv_pred
    sv_pred_dict[patient_id] = sv_pred
    ef_pred_dict[patient_id] = ef_pred

    myocardium_mass_gt_ed[patient_id] = calculate_myocardium_mass(masks[...,0], pixel_spacing, slice_thickness, myocardium_density)
    myocardium_mass_pred_ed[patient_id] = calculate_myocardium_mass(preds[...,0], pixel_spacing, slice_thickness, myocardium_density)
    myocardium_mass_gt_es[patient_id] = calculate_myocardium_mass(masks[...,1], pixel_spacing, slice_thickness, myocardium_density)
    myocardium_mass_pred_es[patient_id] = calculate_myocardium_mass(preds[...,1], pixel_spacing, slice_thickness, myocardium_density)

    # Print per-patient summary
    print(f"\nPatient ID: {patient_id}")
    print(f"  Ground Truth EDV: {edv_gt:.2f} µL, Predicted EDV: {edv_pred:.2f} µL")
    print(f"  Ground Truth ESV: {esv_gt:.2f} µL, Predicted ESV: {esv_pred:.2f} µL")
    print(f"  Ground Truth SV: {sv_gt:.2f} µL, Predicted SV: {sv_pred:.2f} µL")
    print(f"  Ground Truth EF: {ef_gt:.2f} %, Predicted EF: {ef_pred:.2f} %")
    print(f"  Ground Truth Myocardium Mass ED: {myocardium_mass_gt_ed[patient_id]:.2f} mg, Predicted: {myocardium_mass_pred_ed[patient_id]:.2f} mg")
    print(f"  Ground Truth Myocardium Mass ES: {myocardium_mass_gt_es[patient_id]:.2f} mg, Predicted: {myocardium_mass_pred_es[patient_id]:.2f} mg")



# ----------------------------------------
# ✅ Save cardiac volumes to CSV/Excel
# ----------------------------------------
df_volumes = pd.DataFrame({
    "Patient_ID": list(edv_gt_dict.keys()),
    "EDV_GT_µL": [edv_gt_dict[pid] for pid in edv_gt_dict],
    "EDV_Pred_µL": [edv_pred_dict[pid] for pid in edv_pred_dict],
    "ESV_GT_µL": [esv_gt_dict[pid] for pid in esv_gt_dict],
    "ESV_Pred_µL": [esv_pred_dict[pid] for pid in esv_pred_dict],
    "SV_GT_µL": [sv_gt_dict[pid] for pid in sv_gt_dict],
    "SV_Pred_µL": [sv_pred_dict[pid] for pid in sv_pred_dict],
    "EF_GT_%": [ef_gt_dict[pid] for pid in ef_gt_dict],
    "EF_Pred_%": [ef_pred_dict[pid] for pid in ef_pred_dict],
})

volume_csv_path = os.path.join(cleaned_output_dir, "Cardiac_Volumes_After_Island_Removal.csv")
volume_excel_path = os.path.join(cleaned_output_dir, "Cardiac_Volumes_After_Island_Removal.xlsx")

df_volumes.to_csv(volume_csv_path, index=False)
df_volumes.to_excel(volume_excel_path, index=False)

run["Stats/Cardiac_Volumes_After_Island_Removal.csv"].upload(volume_csv_path)
run["Stats/Cardiac_Volumes_After_Island_Removal.xlsx"].upload(volume_excel_path)

print(f"✅ Cardiac volumes saved to:\n  CSV: {volume_csv_path}\n  Excel: {volume_excel_path}")


# ----------------------------------------
# ✅ Save myocardial mass to CSV/Excel
# ----------------------------------------
df_mass = pd.DataFrame({
    "Patient_ID": list(myocardium_mass_gt_ed.keys()),
    "Myocardium_Mass_GT_ED_mg": [myocardium_mass_gt_ed[pid] for pid in myocardium_mass_gt_ed],
    "Myocardium_Mass_Pred_ED_mg": [myocardium_mass_pred_ed[pid] for pid in myocardium_mass_pred_ed],
    "Myocardium_Mass_GT_ES_mg": [myocardium_mass_gt_es[pid] for pid in myocardium_mass_gt_es],
    "Myocardium_Mass_Pred_ES_mg": [myocardium_mass_pred_es[pid] for pid in myocardium_mass_pred_es],
})

mass_csv_path = os.path.join(cleaned_output_dir, "Myocardial_Mass_After_Island_Removal.csv")
mass_excel_path = os.path.join(cleaned_output_dir, "Myocardial_Mass_After_Island_Removal.xlsx")

df_mass.to_csv(mass_csv_path, index=False)
df_mass.to_excel(mass_excel_path, index=False)

run["Stats/Myocardial_Mass_After_Island_Removal.csv"].upload(mass_csv_path)
run["Stats/Myocardial_Mass_After_Island_Removal.xlsx"].upload(mass_excel_path)

print(f"✅ Myocardial mass saved to:\n  CSV: {mass_csv_path}\n  Excel: {mass_excel_path}")






# ----------------------------------------
# ✅ Optional: Save volumes + mass together
# ----------------------------------------
df_combined = pd.concat(
    [df_volumes.set_index("Patient_ID"), df_mass.set_index("Patient_ID")],
    axis=1
).reset_index()

combined_csv_path = os.path.join(cleaned_output_dir, "Cardiac_Volumes_And_Mass.csv")
df_combined.to_csv(combined_csv_path, index=False)
run["Stats/Cardiac_Volumes_And_Mass.csv"].upload(combined_csv_path)

print(f"✅ Combined volumes + mass saved to:\n  CSV: {combined_csv_path}")



# Top-5 errors
metrics_simple = ["EDV", "ESV", "SV", "EF", "Myocardium Mass ED", "Myocardium Mass ES"]
diff_dicts = {
    "EDV": {pid: abs(edv_gt_dict[pid] - edv_pred_dict[pid]) for pid in edv_gt_dict},
    "ESV": {pid: abs(esv_gt_dict[pid] - esv_pred_dict[pid]) for pid in esv_gt_dict},
    "SV": {pid: abs(sv_gt_dict[pid] - sv_pred_dict[pid]) for pid in sv_gt_dict},
    "EF": {pid: abs(ef_gt_dict[pid] - ef_pred_dict[pid]) for pid in ef_gt_dict},
    "Myocardium Mass ED": {pid: abs(myocardium_mass_gt_ed[pid] - myocardium_mass_pred_ed[pid]) for pid in myocardium_mass_gt_ed},
    "Myocardium Mass ES": {pid: abs(myocardium_mass_gt_es[pid] - myocardium_mass_pred_es[pid]) for pid in myocardium_mass_gt_es},
}
dicts_gt = {
    "EDV": edv_gt_dict,
    "ESV": esv_gt_dict,
    "SV": sv_gt_dict,
    "EF": ef_gt_dict,
    "Myocardium Mass ED": myocardium_mass_gt_ed,
    "Myocardium Mass ES": myocardium_mass_gt_es,
}
dicts_pred = {
    "EDV": edv_pred_dict,
    "ESV": esv_pred_dict,
    "SV": sv_pred_dict,
    "EF": ef_pred_dict,
    "Myocardium Mass ED": myocardium_mass_pred_ed,
    "Myocardium Mass ES": myocardium_mass_pred_es,
}

print("\n\n==========  Top 5 Patients by Absolute Error per Metric ==========")
for metric in metrics_simple:
    sorted_errors = sorted(diff_dicts[metric].items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"\n{metric}:")
    for pid, err in sorted_errors:
        gt_val = dicts_gt[metric][pid]
        pred_val = dicts_pred[metric][pid]
        print(f"  {pid}: Error={err:.2f} | GT={gt_val:.2f} | Pred={pred_val:.2f}")

# Statistical summary
metrics = [
    ("EDV", edv_gt_dict, edv_pred_dict, "µL"),
    ("ESV", esv_gt_dict, esv_pred_dict, "µL"),
    ("SV", sv_gt_dict, sv_pred_dict, "µL"),
    ("EF", ef_gt_dict, ef_pred_dict, "%"),
    ("Myocardium Mass ED", myocardium_mass_gt_ed, myocardium_mass_pred_ed, "mg"),
    ("Myocardium Mass ES", myocardium_mass_gt_es, myocardium_mass_pred_es, "mg"),
]

summary_rows = []
for name, gt_dict, pred_dict, unit in metrics:
    gt = list(gt_dict.values())
    pred = list(pred_dict.values())
    diffs = np.subtract(pred, gt)

    mean_gt, std_gt = np.mean(gt), np.std(gt)
    mean_pred, std_pred = np.mean(pred), np.std(pred)
    bias_mean, bias_sd = np.mean(diffs), np.std(diffs)

    shapiro_p = shapiro(diffs).pvalue
    norm_flag = "✅ Normal" if shapiro_p >= 0.05 else "❗ Not normal"

    if shapiro_p >= 0.05:
        tstat, pval = ttest_rel(gt, pred)
        test_str = f"Paired t-test (p={pval:.4f})"
    else:
        tstat, pval = wilcoxon(diffs)
        test_str = f"Wilcoxon (p={pval:.4f})"

    df_tmp = pd.DataFrame({"subj": range(len(gt)), "GT": gt, "Pred": pred})
    df_long = pd.melt(df_tmp, id_vars="subj", value_vars=["GT","Pred"], var_name="method", value_name="val")
    icc2 = intraclass_corr(df_long, targets="subj", raters="method", ratings="val").query("Type=='ICC2'")["ICC"].values[0]

    # rho, _ = spearmanr(gt, pred)
    rho, rho_p = spearmanr(gt, pred)
    pearson_r = np.corrcoef(gt, pred)[0,1]
    r2 = pearson_r**2

    summary_rows.append([
        name,
        f"{mean_gt:.1f} ± {std_gt:.1f}",
        f"{mean_pred:.1f} ± {std_pred:.1f}",
        f"{bias_mean:.1f} ± {bias_sd:.1f}",
        f"{shapiro_p:.4f} ({norm_flag})",
        test_str,
        f"{icc2:.3f}",
        # f"{rho:.3f}",
        f"{rho:.3f} (p={rho_p:.4f})",
        f"{pearson_r:.3f} (r²={r2:.3f})"
    ])

# Print summary
headers = ["Metric", "GT Mean ± SD", "Pred Mean ± SD", "Bias ± SD",
           "Shapiro–Wilk p (Normality)", "Paired Test", "ICC", "Spearman r (p-value)", "Pearson r (r²)"]

summary_str = tabulate(summary_rows, headers=headers, tablefmt="grid")
print("\n===============================")
print("Summary of All Metrics Statistics")
print("===============================\n")
print(summary_str)

# Save to CSV/Excel
df_summary = pd.DataFrame(summary_rows, columns=headers)
csv_path = os.path.join(cleaned_output_dir, "Summary_Combined_Table.csv")
xlsx_path = os.path.join(cleaned_output_dir, "Summary_Combined_Table.xlsx")
df_summary.to_csv(csv_path, index=False)
df_summary.to_excel(xlsx_path, index=False)

# Upload to Neptune
run["Stats/Table/Summary_Combined_Table.csv"].upload(csv_path)
run["Stats/Table/Summary_Combined_Table.xlsx"].upload(xlsx_path)

# Save as text file
txt_path = "Summary_Combined_Table.txt"
timestamp = datetime.datetime.now().isoformat()
with open(txt_path, "w") as f:
    f.write(f"Generated on: {timestamp}\n\n")
    f.write(summary_str)

run["Stats/Table/Summary_Combined_Table.txt"].upload(txt_path)

print("✅ All results saved and uploaded to Neptune.")


In [ ]:
opt: set y limit

In [ ]:
def calculate_volume_from_mask(mask, pixel_spacing, slice_thickness):
    pixel_area = pixel_spacing ** 2
    blood_pool_area = np.sum(mask == 2)
    total_area = blood_pool_area * pixel_area
    volume = total_area * slice_thickness
    return volume

def calculate_myocardium_mass(mask, pixel_spacing, slice_thickness, density):
    pixel_area = pixel_spacing ** 2
    myocardium_area = np.sum(mask == 1)
    total_area = myocardium_area * pixel_area
    myocardium_volume = total_area * slice_thickness
    myocardium_mass = myocardium_volume * density
    return myocardium_mass

def ensure_edv_esv(edv, esv, patient_id, label):
    if esv > edv:
        print(f"Warning: ESV > EDV for patient {patient_id} ({label}). Swapping.")
        return esv, edv
    return edv, esv

def compute_all_stats(gt_values, pred_values):
    import pingouin as pg
    from scipy.stats import shapiro

    r = np.corrcoef(gt_values, pred_values)[0, 1]
    r_squared = r**2

    df = pd.DataFrame({
        "gt": gt_values,
        "pred": pred_values,
        "subject": range(len(gt_values))
    })
    icc_table = pg.intraclass_corr(
        data=df.melt(id_vars=["subject"], value_vars=["gt", "pred"],
                     var_name="method", value_name="score"),
        targets="subject",
        raters="method",
        ratings="score"
    )
    icc2 = icc_table.loc[icc_table["Type"] == "ICC2", "ICC"].values[0]

    diff_vals = np.subtract(pred_values, gt_values)
    _, shapiro_p = shapiro(diff_vals)
    normality = "✅ Normal" if shapiro_p >= 0.05 else "❗ Not normal"

    return {
        "r": r,
        "r_squared": r_squared,
        "icc": icc2,
        "shapiro_p": shapiro_p,
        "normality": normality
    }

# -----------------------------
# Plot Functions
# -----------------------------
def plot_correlation_open(gt_values, pred_values, metric_name, unit, r):
    # concat lists to get global min/max
    x = np.linspace(min(gt_values + pred_values), max(gt_values + pred_values), 100)

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.set_style("white")
    sns.despine()

    sns.scatterplot(
        x=gt_values,
        y=pred_values,
        s=250,
        color='dodgerblue',
        edgecolor='dodgerblue',
        ax=ax,
        zorder=2
    )

    ax.plot(
        x, x,
        color='black',
        linestyle='--',
        linewidth=3,
        zorder=1
    )

    ax.spines['left'].set_linewidth(4)
    ax.spines['bottom'].set_linewidth(4)

    ax.set_xlabel(f'Manual {metric_name} ({unit})', fontsize=21, labelpad=14)
    ax.set_ylabel(f'DL {metric_name} ({unit})', fontsize=21, labelpad=14)
    ax.tick_params(axis='both', labelsize=17)

    # Show Pearson's r instead of r^2
    ax.text(
        0.05, 0.95,
        f'r = {r:.2f}',
        transform=ax.transAxes,
        fontsize=22,
        verticalalignment='top'
    )

    plt.tight_layout()




# Updated Bland–Altman plot function with consistent scaling option
def plot_bland_altman_open(
    gt_values,
    pred_values,
    metric_name,
    unit,
    ax=None,
    fixed_ylim=None,
    text_dx_ratio=0.01,
    font_size=17,
    n_decimals=1
):
    mean_vals = np.mean([gt_values, pred_values], axis=0)
    diff_vals = np.subtract(pred_values, gt_values)

    bias = np.mean(diff_vals)
    std_diff = np.std(diff_vals)
    loa_upper = bias + 1.96 * std_diff
    loa_lower = bias - 1.96 * std_diff

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))

    sns.set_style("white")
    sns.despine()

    sns.scatterplot(
        x=mean_vals,
        y=diff_vals,
        s=250,
        color='red',
        edgecolor='red',
        ax=ax,
        zorder=2
    )

    ax.axhline(bias, color='black', linestyle='-', linewidth=4, zorder=1)
    ax.axhline(loa_upper, color='black', linestyle='--', linewidth=3, zorder=1)
    ax.axhline(loa_lower, color='black', linestyle='--', linewidth=3, zorder=1)

    ax.spines['left'].set_linewidth(4)
    ax.spines['bottom'].set_linewidth(4)

    ax.set_xlabel(f'Bias {metric_name} ({unit})', fontsize=21, labelpad=14)
    ax.set_ylabel(f'Difference in {metric_name} ({unit})', fontsize=21, labelpad=14)
    ax.tick_params(axis='both', labelsize=17)

    # Symmetric and consistent scaling
    if fixed_ylim is not None:
        y_lim = fixed_ylim
    else:
        y_max = max(abs(diff_vals).max(), abs(loa_upper), abs(loa_lower))
        buffer = y_max * 0.1
        y_lim = y_max + buffer

    ax.set_ylim(-y_lim, y_lim)

    x_min, x_max = ax.get_xlim()
    x_pad = (x_max - x_min) * 0.05
    ax.set_xlim(x_min - x_pad, x_max + x_pad)

    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    x_text = ax.get_xlim()[1] + (x_range * text_dx_ratio)

    fmt = f".{n_decimals}f"
    ax.text(x_text, loa_upper, f"+1.96SD:\n{format(loa_upper, fmt)}",
            ha="left", va="center", fontsize=font_size)
    ax.text(x_text, bias, f"Mean:\n{format(bias, fmt)}",
            ha="left", va="center", fontsize=font_size)
    ax.text(x_text, loa_lower, f"-1.96SD:\n{format(loa_lower, fmt)}",
            ha="left", va="center", fontsize=font_size)

    plt.tight_layout()



buffer_ratio = 0.1  # 10% buffer

# Compute differences for EDV and ESV
edv_diff = np.subtract(list(edv_pred_dict.values()), list(edv_gt_dict.values()))
esv_diff = np.subtract(list(esv_pred_dict.values()), list(esv_gt_dict.values()))

# Max differences for EDV and ESV
max_edv_diff = max(abs(edv_diff.max()), abs(edv_diff.min()))
max_esv_diff = max(abs(esv_diff.max()), abs(esv_diff.min()))

# Choose y-axis limits (rounded to nearest 5 µL) with buffer
ylim_edv = np.ceil((max_edv_diff * 1.2) / 5) * 5
ylim_esv = np.ceil((max_esv_diff * 1.2) / 5) * 5

# Ensure ESV scaling is at least half of EDV scaling for comparability
if ylim_esv < ylim_edv * 0.5:
    ylim_esv = ylim_edv * 0.5

# Compute myocardium mass ED scaling
myocardium_ed_diff = np.subtract(list(myocardium_mass_pred_ed.values()), 
                                 list(myocardium_mass_gt_ed.values()))
max_myocardium_ed_diff = max(abs(myocardium_ed_diff.max()), abs(myocardium_ed_diff.min()))
ylim_myocardium_ed = np.ceil((max_myocardium_ed_diff * 1.2) / 5) * 5



# -----------------------------
# Compute all metrics
# -----------------------------
pixel_spacing = 0.1
slice_thickness = 1.0
myocardium_density = 1.05

edv_gt_dict, edv_pred_dict, esv_gt_dict, esv_pred_dict = {}, {}, {}, {}
sv_gt_dict, sv_pred_dict, ef_gt_dict, ef_pred_dict = {}, {}, {}, {}
myocardium_mass_gt_ed, myocardium_mass_pred_ed = {}, {}
myocardium_mass_gt_es, myocardium_mass_pred_es = {}, {}

patient_dirs = [
    os.path.join(test_folder, p)
    for p in os.listdir(test_folder)
    if os.path.isdir(os.path.join(test_folder, p))
]

for patient_dir in patient_dirs:
    patient_id = os.path.basename(patient_dir)
    images_path = os.path.join(patient_dir, "images.npy")
    masks_path = os.path.join(patient_dir, "masks.npy")
    predictions_path = os.path.join(cleaned_predictions_dir, f"{patient_id}_cleaned_predictions.npy")

    if not (os.path.exists(images_path) and os.path.exists(masks_path) and os.path.exists(predictions_path)):
        print(f"Skipping {patient_id}: missing data.")
        continue

    images = np.load(images_path)
    masks = np.load(masks_path)
    predictions = np.load(predictions_path)

    edv_gt, esv_gt, edv_pred, esv_pred = 0, 0, 0, 0

    for slice_idx in range(images.shape[2]):
        m_edv = masks[:, :, slice_idx, 0]
        p_edv = predictions[:, :, slice_idx, 0]
        m_esv = masks[:, :, slice_idx, 1]
        p_esv = predictions[:, :, slice_idx, 1]

        edv_gt += calculate_volume_from_mask(m_edv, pixel_spacing, slice_thickness)
        edv_pred += calculate_volume_from_mask(p_edv, pixel_spacing, slice_thickness)
        esv_gt += calculate_volume_from_mask(m_esv, pixel_spacing, slice_thickness)
        esv_pred += calculate_volume_from_mask(p_esv, pixel_spacing, slice_thickness)

    edv_gt, esv_gt = ensure_edv_esv(edv_gt, esv_gt, patient_id, "GT")
    edv_pred, esv_pred = ensure_edv_esv(edv_pred, esv_pred, patient_id, "Pred")

    sv_gt = edv_gt - esv_gt
    sv_pred = edv_pred - esv_pred
    ef_gt = (sv_gt / edv_gt) * 100
    ef_pred = (sv_pred / edv_pred) * 100

    edv_gt_dict[patient_id] = edv_gt
    esv_gt_dict[patient_id] = esv_gt
    sv_gt_dict[patient_id] = sv_gt
    ef_gt_dict[patient_id] = ef_gt
    edv_pred_dict[patient_id] = edv_pred
    esv_pred_dict[patient_id] = esv_pred
    sv_pred_dict[patient_id] = sv_pred
    ef_pred_dict[patient_id] = ef_pred

    myocardium_mass_gt_ed[patient_id] = calculate_myocardium_mass(masks[..., 0], pixel_spacing, slice_thickness, myocardium_density)
    myocardium_mass_pred_ed[patient_id] = calculate_myocardium_mass(predictions[..., 0], pixel_spacing, slice_thickness, myocardium_density)
    myocardium_mass_gt_es[patient_id] = calculate_myocardium_mass(masks[..., 1], pixel_spacing, slice_thickness, myocardium_density)
    myocardium_mass_pred_es[patient_id] = calculate_myocardium_mass(predictions[..., 1], pixel_spacing, slice_thickness, myocardium_density)

# -----------------------------
# Analysis & Plotting
# -----------------------------
from tabulate import tabulate

metrics = [
    "EDV",
    "ESV",
    "SV",
    "EF",
    "Myocardium Mass ED",
    "Myocardium Mass ES"
]

metric_units = {
    "EDV": "µL",
    "ESV": "µL",
    "SV": "µL",
    "EF": "%",
    "Myocardium Mass ED": "mg",
    "Myocardium Mass ES": "mg"
}

gt_values = [
    list(edv_gt_dict.values()),
    list(esv_gt_dict.values()),
    list(sv_gt_dict.values()),
    list(ef_gt_dict.values()),
    list(myocardium_mass_gt_ed.values()),
    list(myocardium_mass_gt_es.values())
]

pred_values = [
    list(edv_pred_dict.values()),
    list(esv_pred_dict.values()),
    list(sv_pred_dict.values()),
    list(ef_pred_dict.values()),
    list(myocardium_mass_pred_ed.values()),
    list(myocardium_mass_pred_es.values())
]

all_results = {}
for metric_name, gt, pred in zip(metrics, gt_values, pred_values):
    res = compute_all_stats(gt, pred)
    all_results[metric_name] = res

rows = []
for metric_name in metrics:
    res = all_results[metric_name]
    rows.append([
        metric_name,
        f"{res['r']:.3f}",
        f"{res['r_squared']:.3f}",
        f"{res['icc']:.3f}",
        f"{res['shapiro_p']:.4f}",
        res['normality']
    ])

headers = ["Metric", "Pearson r", "r²", "ICC(2,1)", "Shapiro–Wilk p", "Normality"]

print("\n===============================")
print("Summary of All Metrics Statistics")
print("===============================\n")
print(tabulate(rows, headers=headers, tablefmt="grid"))

# -----------------------------
# Plots
# -----------------------------
# -----------------------------
# Plots with optimized Bland–Altman scaling:

ylim_edv = 32                # EDV: to include outliers
ylim_esv = 15                # ESV: tighter data range
ylim_myocardium_ed = 40      # Myocardium Mass ED: aggressive outlier
ylim_ef = 15                 # EF: clinically reasonable range
ylim_myocardium_es = 29      # Myocardium Mass ES: symmetric ±22 mg


for metric_name, gt, pred in zip(metrics, gt_values, pred_values):
    unit = metric_units[metric_name]
    r = all_results[metric_name]['r']

    # Correlation plot
    plot_correlation_open(gt, pred, metric_name, unit, r)
    corr_filename = f"correlation_{metric_name.replace(' ','_').replace('(','').replace(')','')}.png"
    plt.savefig(corr_filename, dpi=300)
    plt.show()
    plt.close()
    run[f"Stats/Plots/{corr_filename}"].upload(corr_filename)

    # Bland–Altman with scaling
    if metric_name == "EDV":
        plot_bland_altman_open(gt, pred, metric_name, unit, fixed_ylim=ylim_edv)
    elif metric_name == "ESV":
        plot_bland_altman_open(gt, pred, metric_name, unit, fixed_ylim=ylim_esv)
    elif metric_name == "Myocardium Mass ED":
        plot_bland_altman_open(gt, pred, metric_name, unit, fixed_ylim=ylim_myocardium_ed)
    elif metric_name == "EF":
        plot_bland_altman_open(gt, pred, metric_name, unit, fixed_ylim=ylim_ef)
    elif metric_name == "Myocardium Mass ES":
        plot_bland_altman_open(gt, pred, metric_name, unit, fixed_ylim=ylim_myocardium_es)
    else:
        plot_bland_altman_open(gt, pred, metric_name, unit)

    bland_filename = f"bland_altman_{metric_name.replace(' ','_').replace('(','').replace(')','')}.png"
    plt.savefig(bland_filename, dpi=300)
    plt.show()
    plt.close()
    run[f"Stats/Plots/{bland_filename}"].upload(bland_filename)




In [ ]:
Combine Correlation & BA plot in 1 figure (5x2) 

In [ ]:
import matplotlib.image as mpimg

run.wait()

metrics = [
    'End-Diastolic Volume (EDV)',
    'End-Systolic Volume (ESV)',
    'Ejection Fraction (EF)',
    'Myocardium Mass (ED)',
    'Myocardium Mass (ES)',
]

bland_files = [
    "bland_altman_EDV.png",
    "bland_altman_ESV.png",
    "bland_altman_EF.png",
    "bland_altman_Myocardium_Mass_ED.png",
    "bland_altman_Myocardium_Mass_ES.png",
]

corr_files = [
    "correlation_EDV.png",
    "correlation_ESV.png",
    "correlation_EF.png",
    "correlation_Myocardium_Mass_ED.png",
    "correlation_Myocardium_Mass_ES.png",
]

fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(18, 32), dpi=300)

row_title_positions = [0.922, 0.737, 0.552, 0.367, 0.182]

plt.tight_layout(rect=[0, 0, 1, 0.96], h_pad=10)

for i in range(5):
    # Downloads from your actual folder structure
    run[f"Stats/Plots/{bland_files[i]}"].download(bland_files[i])
    run[f"Stats/Plots/{corr_files[i]}"].download(corr_files[i])

    ba_img = mpimg.imread(bland_files[i])
    corr_img = mpimg.imread(corr_files[i])

    axes[i, 0].imshow(ba_img)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(corr_img)
    axes[i, 1].axis('off')

    fig.text(
        0.5,
        row_title_positions[i],
        metrics[i],
        ha='center',
        va='bottom',
        fontsize=18,
        fontweight='bold'
    )

fig.suptitle("Manual vs DL (n = 20)", fontsize=26, fontweight='semibold', y=0.96)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.subplots_adjust(wspace=0.06)

final_output = "comparison_5x2_combined_clean.png"
plt.savefig(final_output, dpi=300)
plt.show()

run["Figures/Combined_5x2_Metrics"].upload(final_output)
print("✅ Uploaded to Neptune: Figures/Combined_5x2_Metrics")


In [ ]:
6x2

In [ ]:
# Local save directory
local_dir = "/workspaces/PhD/outputs/SEG420/BA & Cor"
os.makedirs(local_dir, exist_ok=True)

run.wait()

# Updated metric list to include SV
metrics = [
    'End-Diastolic Volume (EDV)',
    'End-Systolic Volume (ESV)',
    'Stroke Volume (SV)',  # ADDED
    'Ejection Fraction (EF)',
    'Myocardium Mass (ED)',
    'Myocardium Mass (ES)',
]

# Updated file name lists to include SV plots
bland_files = [
    "bland_altman_EDV.png",
    "bland_altman_ESV.png",
    "bland_altman_SV.png",  # ADDED
    "bland_altman_EF.png",
    "bland_altman_Myocardium_Mass_ED.png",
    "bland_altman_Myocardium_Mass_ES.png",
]

corr_files = [
    "correlation_EDV.png",
    "correlation_ESV.png",
    "correlation_SV.png",  # ADDED
    "correlation_EF.png",
    "correlation_Myocardium_Mass_ED.png",
    "correlation_Myocardium_Mass_ES.png",
]

# Increased nrows to 6 for the additional SV row
fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(18, 34), dpi=300)

# Updated title positions
row_title_positions = [0.922, 0.768, 0.614, 0.460, 0.306, 0.152]

# Adjust subplot spacing
plt.subplots_adjust(hspace=0.7, wspace=0.06)

for i in range(6):
    # Download from Neptune
    run[f"Stats/Plots/{bland_files[i]}"].download(bland_files[i])
    run[f"Stats/Plots/{corr_files[i]}"].download(corr_files[i])

    # Save a local copy in the requested folder
    local_ba_path = os.path.join(local_dir, bland_files[i])
    local_corr_path = os.path.join(local_dir, corr_files[i])
    os.replace(bland_files[i], local_ba_path)
    os.replace(corr_files[i], local_corr_path)

    # Load from local path for plotting
    ba_img = mpimg.imread(local_ba_path)
    corr_img = mpimg.imread(local_corr_path)

    axes[i, 0].imshow(ba_img)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(corr_img)
    axes[i, 1].axis('off')

    # Add subtitle for each row (centered)
    fig.text(
        0.5,
        row_title_positions[i],
        metrics[i],
        ha='center',
        va='bottom',
        fontsize=18,
        fontweight='bold'
    )

# Main title
fig.suptitle("Ground Truth vs DL (n = 15)", fontsize=26, fontweight='semibold', y=0.96)

# Layout adjustment
plt.tight_layout(rect=[0, 0, 1, 0.94])

# Save combined figure
final_output = "comparison_6x2_combined_1.png"
plt.savefig(final_output, dpi=300)  # original location
local_combined_path = os.path.join(local_dir, final_output)
plt.savefig(local_combined_path, dpi=300)  # local copy
plt.show()

# Upload to Neptune
run["Figures/Combined_6x2_Metrics_1"].upload(final_output)
print("Uploaded to Neptune: Figures/Combined_6x2_Metrics")
print(f"All plots saved locally in: {local_dir}")


In [ ]:
DICE & Volume Error/Mass Error!!

In [ ]:
import pandas as pd
from scipy.stats import pearsonr
from scipy import stats


# ---------------------------------
# 1️⃣ Load volume and myocardial mass data
# ---------------------------------
df_volumes = pd.read_csv(
    "/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal/Cardiac_Volumes_After_Island_Removal.csv"
)

df_mass = pd.read_csv(
    "/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal/Myocardial_Mass_After_Island_Removal.csv"
)

# ---------------------------------
# 2️⃣ Compute volume errors
# ---------------------------------
df_volumes["EDV_Error"] = abs(df_volumes["EDV_GT_µL"] - df_volumes["EDV_Pred_µL"])
df_volumes["ESV_Error"] = abs(df_volumes["ESV_GT_µL"] - df_volumes["ESV_Pred_µL"])

# ---------------------------------
# 3️⃣ Compute myocardial mass errors
# ---------------------------------
df_mass["MyocardialMass_ED_Error"] = abs(
    df_mass["Myocardium_Mass_GT_ED_mg"] - df_mass["Myocardium_Mass_Pred_ED_mg"]
)
df_mass["MyocardialMass_ES_Error"] = abs(
    df_mass["Myocardium_Mass_GT_ES_mg"] - df_mass["Myocardium_Mass_Pred_ES_mg"]
)

# ---------------------------------
# 4️⃣ Load per-frame Dice scores
# ---------------------------------
df_dice_per_frame = pd.read_csv(
    "/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal/SciPy_dice_scores_per_frame.csv"
)

# Pivot Dice scores per phase
dice_pivot = df_dice_per_frame.pivot(
    index="Patient_ID",
    columns="Frame_Type",
    values=["BloodPool_Dice", "Myocardium_Dice"]
).reset_index()

# Rename columns for clarity
dice_pivot.columns = [
    "Patient_ID",
    "BloodPool_Dice_Diastole",
    "BloodPool_Dice_Systole",
    "Myocardium_Dice_Diastole",
    "Myocardium_Dice_Systole"
]

# ---------------------------------
# 5️⃣ Merge all into a single dataframe
# ---------------------------------
df_all = df_volumes[["Patient_ID", "EDV_Error", "ESV_Error"]].merge(
    df_mass[["Patient_ID", "MyocardialMass_ED_Error", "MyocardialMass_ES_Error"]],
    on="Patient_ID"
).merge(
    dice_pivot,
    on="Patient_ID"
)

# ---------------------------------
# 6️⃣ Compute correlations
# ---------------------------------
results = []

metrics = [
    ("EDV_Error", "BloodPool_Dice_Diastole"),
    ("ESV_Error", "BloodPool_Dice_Systole"),
    ("MyocardialMass_ED_Error", "Myocardium_Dice_Diastole"),
    ("MyocardialMass_ES_Error", "Myocardium_Dice_Systole")
]

print("\n========== Correlation Results (Dice vs Volume Error & Dice vs Myocardial Mass Error)==========\n")
print(f"{'Error Metric':30} {'Dice':30} {'r':>8} {'R²':>8}")
print("-" * 80)

for error_col, dice_col in metrics:
    r, p = pearsonr(df_all[dice_col], df_all[error_col])
    r2 = r**2
    print(f"{error_col:30} {dice_col:30} {r:8.2f} {r2:8.2f}")
    results.append({
        "Error Metric": error_col,
        "Dice": dice_col,
        "r": r,
        "R2": r2,
        "p_value": p
    })

# ---------------------------------
# 7️⃣ Save results as CSV
# ---------------------------------
df_results = pd.DataFrame(results)
df_results.to_csv("Dice_vs_Error_Correlations.csv", index=False)

print("\n✅ Results saved to Dice_vs_Error_Correlations.csv.")



In [ ]:
# ---------------------------------
# 1️⃣ Load CSVs
# ---------------------------------
df_volumes = pd.read_csv(
    "/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal/Cardiac_Volumes_After_Island_Removal.csv"
)
df_mass = pd.read_csv(
    "/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal/Myocardial_Mass_After_Island_Removal.csv"
)
df_dice_per_frame = pd.read_csv(
    "/workspaces/PhD/outputs/SEG420/Test_DiceScore_After_Island_Removal/SciPy_dice_scores_per_frame.csv"
)

# ---------------------------------
# 2️⃣ Compute errors
# ---------------------------------
df_volumes["EDV_Error"] = abs(df_volumes["EDV_GT_µL"] - df_volumes["EDV_Pred_µL"])
df_volumes["ESV_Error"] = abs(df_volumes["ESV_GT_µL"] - df_volumes["ESV_Pred_µL"])

df_mass["MyocardialMass_ED_Error"] = abs(
    df_mass["Myocardium_Mass_GT_ED_mg"] - df_mass["Myocardium_Mass_Pred_ED_mg"]
)
df_mass["MyocardialMass_ES_Error"] = abs(
    df_mass["Myocardium_Mass_GT_ES_mg"] - df_mass["Myocardium_Mass_Pred_ES_mg"]
)

# ---------------------------------
# 3️⃣ Pivot Dice per phase
# ---------------------------------
dice_pivot = df_dice_per_frame.pivot(
    index="Patient_ID",
    columns="Frame_Type",
    values=["BloodPool_Dice", "Myocardium_Dice"]
).reset_index()

dice_pivot.columns = [
    "Patient_ID",
    "BloodPool_Dice_Diastole",
    "BloodPool_Dice_Systole",
    "Myocardium_Dice_Diastole",
    "Myocardium_Dice_Systole"
]

# ---------------------------------
# 4️⃣ Merge all into single DataFrame
# ---------------------------------
df_all = df_volumes[["Patient_ID", "EDV_Error", "ESV_Error"]].merge(
    df_mass[["Patient_ID", "MyocardialMass_ED_Error", "MyocardialMass_ES_Error"]],
    on="Patient_ID"
).merge(
    dice_pivot,
    on="Patient_ID"
)

# ---------------------------------
# 5️⃣ Correlation and plots
# ---------------------------------
metrics = [
    ("EDV_Error", "BloodPool_Dice_Diastole"),
    ("ESV_Error", "BloodPool_Dice_Systole"),
    ("MyocardialMass_ED_Error", "Myocardium_Dice_Diastole"),
    ("MyocardialMass_ES_Error", "Myocardium_Dice_Systole")
]

r_values = {}
os.makedirs("Dice_vs_Error_Plots", exist_ok=True)

print("\n\n========== Correlation Plots ==========\n")

for error_col, dice_col in metrics:
    plt.figure(figsize=(8,6))
    sns.regplot(
        x=dice_col,
        y=error_col,
        data=df_all,
        ci=95,
        scatter_kws={"s": 60, "color": "blue", "edgecolor": "black"},
        line_kws={"color": "red"}
    )
    r, _ = stats.pearsonr(df_all[dice_col], df_all[error_col])
    r_squared = r**2
    r_values[error_col] = r

    plt.xlabel(f"{dice_col}", fontsize=14)
    plt.ylabel(error_col, fontsize=14)
    plt.title(f"Correlation: {dice_col} vs {error_col}", fontsize=16)
    plt.annotate(
        f"$R^2$ = {r_squared:.2f}",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=12,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.5)
    )
    plt.grid(True)
    plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.15)

    plot_path = f"Dice_vs_Error_Plots/{dice_col}_vs_{error_col}.png"
    plt.savefig(plot_path)
    plt.show()
    plt.close()

# ---------------------------------
# 6️⃣ Print summary
# ---------------------------------
# print("\n\n========== Correlation Summary ==========\n")
print("\n\n========== Correlation Summary (Dice vs Volume Error & Dice vs Myocardial Mass Error) ==========\n")
print(f"{'Error Metric':30} {'Dice':30} {'r':>8} {'R²':>8}")
print("-"*80)
for error_col, dice_col in metrics:
    r = r_values[error_col]
    r_squared = r**2
    if abs(r) > 0.7:
        interp = "Strong correlation"
    elif abs(r) > 0.4:
        interp = "Moderate correlation"
    elif abs(r) > 0.2:
        interp = "Weak correlation"
    else:
        interp = "No clear correlation"

    print(f"{error_col:30} {dice_col:30} {r:8.2f} {r_squared:8.2f}  {interp}")

print("\n✅ All plots saved to Dice_vs_Error_Plots/")

In [ ]:
run.stop()